# RANDOM FOREST

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
import os
from sklearn.feature_selection import SelectFromModel
import random
import logging
from IPython.display import display


In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error

# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# lists previously defined in 05_0_preproc_helpers.ipynb
categorical_features = ['Brand', 'model', 'transmission', 'fuelType']              
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)


X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


__Some notes:__

The following preprocessing steps have already been applied to full_train_dataset (in the 05_0_preproc_helpers notebook):
- The columns carID and hasDamage were removed.
- The functions fill_unknown, basic_string_transformer and correct_invalid_brands_in_df were applied across the dataset.
- As a result, the categorical columns no longer contain missing values (all of the previously missing entries are marked as "UNKNOWN"), and all categorical values have been normalized: uppercase, accent-free, strictly alphanumeric and without spaces. Brand values that partially matched a valid brand name have also been corrected.
- Numerical columns may still contain missing values and categorical features still require additional processing, which will be handled using a proper fit/transform logic during cross-validation to avoid data leakage.

## 2. Randomized Hyperparameter Search with K-Fold Cross-Validation

This section defines and executes a full randomized hyperparameter search for a Random Forest regressor using an 8-fold K-Fold cross-validation strategy. A search space of candidate hyperparameters is sampled (n_iter = 50), and for each sampled configuration the following steps are performed inside every fold:

Train/validation split according to the KFold indices.

Numerical preprocessing using a strict fit/transform logic: median-based imputations, domain-aware imputers for year, mileage, engineSize, tax, mpg, paintQuality%, and previousOwners.

Categorical cleaning and harmonization, resolving invalid or ambiguous brands, models, transmissions, and fuel types, based only on the training split.

Categorical encoding: target encoding for high-cardinality features (Brand, model) and one-hot encoding for the remaining categorical features.

Numerical scaling using StandardScaler (fit on train, transform on validation).

Feature assembly by concatenating normalized categorical features with the scaled numerical features.

Model training and evaluation with the current Random Forest hyperparameters, computing RMSE, MAE, and R² for both the training and validation sets.

Metrics from all folds are aggregated to produce mean RMSE, MAE, and a combined score (0.5·RMSE + 0.5·MAE). The system keeps track of the best-performing configurations under each criterion. All results are logged to a file and stored in a summary dataframe for inspection.

## 3. Model Selection Experiments (Random Search and 8-Fold CV)  <a id="expruns"></a>

### 3.1. Full Feature Set (no Feature Engineering and no Feature Selection) <a id="allfeat"></a>

- In this section, we performed a random search to check wether the chosen hyperparameters search space was reasonable and to establish baseline results for the next improvements.

In [2]:
# set core configuration for cross validation
N_SPLITS = 8 # number of folds used in k-fold cross validation
RANDOM_STATE = 42

# creation of the KFold object that will generate the train/validation splits
kf = KFold(
    n_splits=N_SPLITS, # number of folds, in this case will be 8 folds
    shuffle=True,  # shuffle data before splitting to make folds more balanced
    random_state=RANDOM_STATE # using the fixed seed 
)

# ------ // ------
# Definition of the hyperparameters that will be tested in the Randomized Search

param_distributions = {
    # number of trees in the forest: 
    "n_estimators": [200, 400, 600, 800, 1000], 
    # maximum depth of each tree:
    "max_depth": [5, 7, 9, 11, 15, 20],
    # minimum number of samples required to split an internal node:
    "min_samples_split": [2, 4, 6, 10],
    # minimum number of samples required to be at a leaf node:
    "min_samples_leaf": [1, 2, 4, 8],
    # number of features to consider when looking for the best split:
    "max_features": [None], # using all features
    "bootstrap": [True, False], # whether bootstrap samples are used when building trees
}

N_RANDOM_CONFIGS = 20 # number of random different hyperparameter configurations to try


# this sampler will generate random combinations of hyperparameters
# from the previously defined search space 
sampler = ParameterSampler(
    param_distributions, # the dictionary with the hyperparameter search space
    n_iter=N_RANDOM_CONFIGS, # number of different random configurations to generate
    random_state=RANDOM_STATE 
)

search_results = [] # list that will store summary results for each hyperparameter configuration

# best configuration according to validation rmse
best_rmse = np.inf # our initial best rmse is infinite so any real rmse will be smaller
best_config = None # will store the params corresponding to the best rmse

# best configuration according to validation mae
best_mae = np.inf
best_config_mae = None

# best configuration according to a combined metric 0.5 * rmse + 0.5 * mae on validation
best_combo = np.inf
best_config_combo = None

# we have a log file to store detailed logs of the random search process
# this is useful for later analysis of the results, to avoid losing information if the notebook crashes
# and following the progress of the search
log_path = "rf_random_search_log_.txt"

# opens the log file for writing
with open(log_path, "w", encoding="utf-8") as log_file:

    # this is a helper function to write messages to both console and log file
    def log(msg: str):
        print(msg) # print to stdout so progress is visible during execution
        log_file.write(msg + "\n") # also write the same message to the log file
        log_file.flush() # flush so log is updated immediately on disk

    log("# starting random forest")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")

    # -----------------------------
    # main loop of the random search with cross validation
    # each iteration uses one sampled hyperparameter configuration
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f" - CONFIG {config_id}/{N_RANDOM_CONFIGS} ")
        log(f"Parameters: {params}")

        # lists to store validation metrics per fold for this configuration
        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        # lists to store training metrics per fold for this configuration
        fold_rmses_train = []
        fold_maes_train  = []
        fold_r2s_train   = []

        # cross validation loop for the current hyperparameter configuration
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] -- FOLD {fold}/{N_SPLITS}")

            # ---------------------------------
            # 1) split the data into train and validation for the current fold
            # ---------------------------------
            X_train = X.iloc[train_idx].copy() # training features for the current fold
            X_val   = X.iloc[val_idx].copy() # validation features for the current fold
            y_train = y.iloc[train_idx].copy() # training targets for the current fold
            y_val   = y.iloc[val_idx].copy() # validation targets for the current fold

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # log missing values BEFORE imputations
            # --> for the numeric features
            log(f"[C{config_id}|F{fold}] NaNs before (just numerical):")
            log(str(X_train[numeric_features].isna().sum()))
            # --> for the categorical features (we are not supposed to have NaNs here because they were already temporarily handled as UNKNOWN, but just in case)
            log(f"[C{config_id}|F{fold}] NaNs before (categorical):")
            log(str(X_train[categorical_features].isna().sum()))
            # count how many "UNKNOWN" exist in the categorical features before further processing
            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' before (categorical):")
            log(str(unknown_counts_before))

            # ---------------------------------
            # 2) numerical preprocessing 
            #    using fit on train and transform on val
            #    each "state" object stores statistics or rules learned from train
            # ---------------------------------

            # 2.1) YEAR 
            # impute year based on median year per model learned from training data
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2) MILEAGE
            # impute mileage using a custom imputer that can also enforce absolute values
            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3) ENGINE SIZE
            # impute engine size using a imputer fitted on the training data
            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)


            # 2.5) MPG
            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            
            # 2.7) PREVIOUS OWNERS
            owners_state = fit_previous_owners_imputer(
                X_train,
                owners_col="previousOwners",
                year_col="year",
                mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            

            # ---------------------------------
            # 2.8) BRAND 
            # ---------------------------------
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )
            X_train, brand_corr_train, brand_still_invalid_train = transform_ambiguous_brands(
                X_train,
                brand_state,
            )
            X_val, brand_corr_val, brand_still_invalid_val = transform_ambiguous_brands(
                X_val,
                brand_state,
            )
            

            # ---------------------------------
            # 2.9) MODEL
            # ---------------------------------
            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )
            X_train, model_corr_train, model_still_invalid_train = transform_invalid_models(
                X_train,
                model_state,
            )
            X_val, model_corr_val, model_still_invalid_val = transform_invalid_models(
                X_val,
                model_state,
            )
             

            # ---------------------------------
            # 2.10) TRANSMISSION
            # ---------------------------------
            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )
            X_train, transm_corr_train, transm_still_invalid_train = transform_transmission_resolver(
                X_train,
                transm_state,
            )
            X_val, transm_corr_val, transm_still_invalid_val = transform_transmission_resolver(
                X_val,
                transm_state,
            )
            
            
            # ---------------------------------
            # 2.11) FUEL TYPE
            # ---------------------------------
            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )
            X_train, fuel_corr_train, fuel_still_invalid_train = transform_fueltype_resolver(
                X_train,
                fuel_state,
            )
            X_val, fuel_corr_val, fuel_still_invalid_val = transform_fueltype_resolver(
                X_val,
                fuel_state,
            )
            # --
            log(f"[C{config_id}|F{fold}] Após resolver fuelType:")
            log("  Valores distintos (train): " +
                str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))
            log("  Ainda problemáticos (train): " + str(fuel_still_invalid_train[:10]))
            # --

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize") 

            # ---------------------------------
            # Some extra logs for checking if the preprocessing went well
            # ---------------------------------
            log(f"[C{config_id}|F{fold}] --- Summary ---")
            num_means_train = X_train[numeric_features].mean()
            num_nans_train  = X_train[numeric_features].isna().sum()
            log("  numerical means (train):")
            log(str(num_means_train))
            log("  numerical NaNs  (train):")
            log(str(num_nans_train))

            num_means_val = X_val[numeric_features].mean()
            num_nans_val  = X_val[numeric_features].isna().sum()
            log("  numerical means  (val):")
            log(str(num_means_val))
            log("  numerical NaNs   (val):")
            log(str(num_nans_val))

            log(f"[C{config_id}|F{fold}] --- categorical summary ---")
            cat_nans_train     = X_train[categorical_features].isna().sum()
            cat_unknown_train  = (X_train[categorical_features] == "UNKNOWN").sum()
            log("  NaNs per categorical (train):")
            log(str(cat_nans_train))
            log("  'UNKNOWN' per categorical  (train):")
            log(str(cat_unknown_train))

            log(f"[C{config_id}|F{fold}] distinct values in categorical (train):")
            for col in categorical_features:
                uniques = sorted(X_train[col].dropna().astype(str).unique())
                if len(uniques) > 30:
                    shown = uniques[:30]
                    log(f"    {col}: {len(uniques)} distinct values. First 30: {shown} ...")
                else:
                    log(f"    {col}: {len(uniques)} distinct values: {uniques}")

            # left UNKNOWN 
            unknown_mask_train = (X_train[categorical_features] == "UNKNOWN")
            unknown_mask_val   = (X_val[categorical_features]   == "UNKNOWN")

            unknown_counts_train = unknown_mask_train.sum()
            unknown_counts_val   = unknown_mask_val.sum()

            rows_with_unknown_train = unknown_mask_train.any(axis=1).sum()
            rows_with_unknown_val   = unknown_mask_val.any(axis=1).sum()

            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after (categorical, train):")
            log(str(unknown_counts_train))
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after  (categorical, val):")
            log(str(unknown_counts_val))
            log(f"[C{config_id}|F{fold}] Num of rows with >=1 UNKNOWN (train): {rows_with_unknown_train}")
            log(f"[C{config_id}|F{fold}] Num of rows with >=1 UNKNOWN (val)  : {rows_with_unknown_val}")


            # ---------------------------------
            # 3) categorical encoding
            #    (Target encoding for Brand+model, One-Hot for the rest)
            # ---------------------------------
            high_card_features = ["Brand", "model"]  # features with high cardinality => treatment with target encoding
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            # --
            log(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")
            # --

            # 3.1) Target Encoding para Brand e model
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)

            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-Hot Encoding para as restantes categóricas
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])

            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")


            unknown_cols_train = [col for col in X_train_cat.columns if "UNKNOWN" in str(col)]
            unknown_cols_val   = [col for col in X_val_cat.columns if "UNKNOWN" in str(col)]



            # ---------------------------------
            # 4) NUMERICAL SCALING
            # ---------------------------------
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features]) # fit on train and transform train
            X_val_num   = scaler.transform(X_val[numeric_features]) # transform validation using same scaler

            # convert numpy arrays back to dataframes with meaningful column names and consistent indices
            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features]
            )

            log(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # ---------------------------------
            # 5) concatenate scaled numeric and encoded categorical features
            # ---------------------------------
            X_train_final = pd.concat(
                [X_train_num_df, X_train_cat],
                axis=1
            )
            X_val_final = pd.concat(
                [X_val_num_df, X_val_cat],
                axis=1
            )

            log(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # ---------------------------------
            # 6) instantiate and train the random forest regressor for this configuration
            # ---------------------------------
            rf = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1, # uses all available cores 
                **params # current hyperparameter configuration
            )

            log(f"[C{config_id}|F{fold}] training RandomForestRegressor...")
            rf.fit(X_train_final, y_train) # train the model on the fully processed training data

            # generate predictions on training and validation sets for metric computation
            y_pred_train = rf.predict(X_train_final)
            y_pred_val   = rf.predict(X_val_final)

            # ---------------------------------
            # 7) compute metrics per fold for both train and validation
            #    used to monitor performance and detect potential overfitting/underfitting
            # ---------------------------------
            # Validation metrics:
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)

            # TRAIN metrics: 
            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)

            log(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.4f} | MAE: {mae_tr:.4f} | R²: {r2_tr:.4f}")
            log(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R²: {r2_val:.4f}")

            # store the validation metrics for this fold
            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)

            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)

        # average metrics of all folds for this configuration
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)

        mean_rmse_tr = np.mean(fold_rmses_train)
        mean_mae_tr  = np.mean(fold_maes_train)
        mean_r2_tr   = np.mean(fold_r2s_train)

        # combined score giving equal weight to rmse and mae on validation
        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - TRAIN RMSE mean: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f} | R²: {mean_r2_tr:.4f}")
        log(f"Config {config_id} - VAL   RMSE mean: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | R²: {mean_r2_val:.4f}")
        log(f"Config {config_id} - Score combined (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.4f}")


        # store a record with configuration, metrics and combined score to the search results list
        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "combo_score": combo_score,
        })

        # update (or not) RMSE (VAL)
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id} w/ mean RMSE (VAL) = {best_rmse:.4f}")

        # update (or not) MAE (VAL)
        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id} w/ mean MAE  (VAL) = {best_mae:.4f}")

        # update (or not) best combined score (VAL)
        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED] Config {config_id} w/ score = {best_combo:.4f}")


    # after iterating over all sampled configurations, log a summary of the best ones
    log("")
    log("")
    log("# =============================")
    log("# END")
    log("# =============================")
    log(f"Best config (min RMSE VAL): {best_config}")
    log(f"best mean RMSE  (VAL): {best_rmse:.4f}")
    log(f"Best config (min MAE VAL): {best_config_mae}")
    log(f"BEST mean MAE (VAL): {best_mae:.4f}")
    log(f"best config (combined score VAL): {best_config_combo}")
    log(f"best combined score (VAL): {best_combo:.4f}")

# -----------------------------
# final summary of the random search (as a dataframe)
# -----------------------------
results_df = pd.DataFrame(search_results) # convert list of dicts to dataframe
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True) # sort by validation rmse

display(results_df_sorted.head(10)) # configurations with lowest validation rmse

# print the best configurations according to each criterion
print("\nBest config (min RMSE VAL):")
print(best_config)
print("best mean RMSE (VAL):", best_rmse)

print("\nbest config (min MAE VAL):")
print(best_config_mae)
print("best mean MAE  (VAL):", best_mae)

print("\nbest config found (combined score VAL = 0.5*RMSE + 0.5*MAE):")
print(best_config_combo)
print("best combined score (VAL):", best_combo)

print(f"\nLogs stored at: {log_path}")


# starting random forest
N_SPLITS = 8, N_RANDOM_CONFIGS = 20
param_distributions = {'n_estimators': [200, 400, 600, 800, 1000], 'max_depth': [5, 7, 9, 11, 15, 20], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 4, 8], 'max_features': [None], 'bootstrap': [True, False]}

 - CONFIG 1/20 
Parameters: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 15, 'bootstrap': False}

[CONFIG 1] -- FOLD 1/8
[C1|F1] X_train shape: (66476, 10), X_val shape: (9497, 10)
[C1|F1] y_train shape: (66476,), y_val shape: (9497,)
[C1|F1] NaNs before (just numerical):
year              1306
mileage           1252
engineSize        1344
tax               6940
mpg               6961
previousOwners    1331
dtype: int64
[C1|F1] NaNs before (categorical):
Brand           0
model           0
transmission    0
fuelType        0
dtype: int64
[C1|F1] 'UNKNOWN' before (categorical):
Brand           1328
model           1314
transmission    1944
fuelTy

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_train_mean,mae_train_mean,r2_train_mean,rmse_mean,mae_mean,r2_mean,combo_score
7,8,1000,6,1,None,15,True,1560.356118,999.831279,0.974317,2262.761511,1336.104531,0.945830,1799.433021
2,3,200,6,2,None,15,True,1654.235643,1016.679770,0.971132,2302.755387,1340.440144,0.943885,1821.597765
1,2,600,10,8,None,20,True,2071.908363,1152.730895,0.954716,2452.548694,1384.109758,0.936353,1918.329226
17,18,1000,2,8,None,20,True,2072.037323,1152.623896,0.954710,2453.685894,1384.229266,0.936296,1918.957580
5,6,400,2,2,None,11,True,2104.876773,1363.980034,0.953265,2481.456725,1511.768869,0.934906,1996.612797
0,1,400,10,2,None,15,False,1833.286328,1133.884158,0.964532,2688.265474,1541.592787,0.923514,2114.929130
13,14,200,10,8,None,20,False,2111.611735,1195.377448,0.952962,2730.215599,1544.515913,0.920979,2137.365756
9,10,800,10,8,None,15,False,2200.235542,1282.587178,0.948932,2734.513206,1554.392672,0.920733,2144.452939
4,5,800,2,4,None,20,False,1730.254047,977.062641,0.968414,2747.313085,1552.029966,0.920200,2149.671526
6,7,200,10,4,None,9,True,2555.086939,1626.459315,0.931138,2755.067827,1700.358886,0.919803,2227.713357



Best config (min RMSE VAL):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
best mean RMSE (VAL): 2262.7615106654034

best config (min MAE VAL):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
best mean MAE  (VAL): 1336.1045313730594

best config found (combined score VAL = 0.5*RMSE + 0.5*MAE):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
best combined score (VAL): 1799.4330210192315

Logs stored at: rf_random_search_log_.txt


#### 3.1.2 With FS

### 3.2. Feature Ablation: Excluding `previousOwners` <a id="noprevown"></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, ParameterSampler
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------------------------------------------------
# 0) FEATURES
# ---------------------------------------------------------
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg']  # we removed previousOwners feature from here
categorical_features = ["Brand", "model", "transmission", "fuelType"]

# IMPORTANT: baseline copy to avoid leaking state between folds/configs
BASE_NUMERIC_FEATURES = numeric_features.copy()

# set core configuration for cross validation
N_SPLITS = 8  # number of folds used in k-fold cross validation
RANDOM_STATE = 42

# creation of the KFold object that will generate the train/validation splits
kf = KFold(
    n_splits=N_SPLITS,  # number of folds, in this case will be 8 folds
    shuffle=True,  # shuffle data before splitting to make folds more balanced
    random_state=RANDOM_STATE  # using the fixed seed
)

# ------ // ------
# Definition of the hyperparameters that will be tested in the Randomized Search
param_distributions = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [7, 9, 11, 15, 20],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "max_features": ["sqrt", 0.33, 0.5, 1.0],
    "bootstrap": [True],
}

N_RANDOM_CONFIGS = 15  # number of random different hyperparameter configurations to try

# this sampler will generate random combinations of hyperparameters
# from the previously defined search space
sampler = ParameterSampler(
    param_distributions,  # the dictionary with the hyperparameter search space
    n_iter=N_RANDOM_CONFIGS,  # number of different random configurations to generate
    random_state=RANDOM_STATE
)

search_results = []  # list that will store summary results for each hyperparameter configuration

# best configuration according to validation rmse
best_rmse = np.inf  # our initial best rmse is infinite so any real rmse will be smaller
best_config = None  # will store the params corresponding to the best rmse

# best configuration according to validation mae
best_mae = np.inf
best_config_mae = None

# best configuration according to a combined metric 0.5 * rmse + 0.5 * mae on validation
best_combo = np.inf
best_config_combo = None

# we have a log file to store detailed logs of the random search process
log_path = "rf_random_search_log_2_.txt"

# opens the log file for writing
with open(log_path, "w", encoding="utf-8") as log_file:

    # this is a helper function to write messages to both console and log file
    def log(msg: str):
        print(msg)  # print to stdout so progress is visible during execution
        log_file.write(msg + "\n")  # also write the same message to the log file
        log_file.flush()  # flush so log is updated immediately on disk

    log("# starting random forest")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")
    log("Target transform: log1p(price) during training; metrics computed back in euros via expm1(pred).")
    log("Note: previousOwners is not used (excluded from numeric_features and dropped per fold).")

    # -----------------------------
    # main loop of the random search with cross validation
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f" - CONFIG {config_id}/{N_RANDOM_CONFIGS} ")
        log(f"Parameters: {params}")

        # lists to store validation metrics per fold for this configuration
        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        # lists to store training metrics per fold for this configuration
        fold_rmses_train = []
        fold_maes_train  = []
        fold_r2s_train   = []

        # cross validation loop for the current hyperparameter configuration
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] -- FOLD {fold}/{N_SPLITS}")

            # LOCAL numeric feature list for this fold (prevents state leaking between folds)
            numeric_features_fold = BASE_NUMERIC_FEATURES.copy()

            # ---------------------------------
            # 1) split the data into train and validation for the current fold
            # ---------------------------------
            X_train = X.iloc[train_idx].copy()  # training features for the current fold
            X_val   = X.iloc[val_idx].copy()    # validation features for the current fold
            y_train = y.iloc[train_idx].copy()  # training targets for the current fold (EUROS)
            y_val   = y.iloc[val_idx].copy()    # validation targets for the current fold (EUROS)

            # EXTRA GUARANTEE: drop previousOwners so it can never leak into the final matrix
            X_train = X_train.drop(columns=["previousOwners"], errors="ignore")
            X_val   = X_val.drop(columns=["previousOwners"], errors="ignore")

            # LOG TARGET (training happens in log space)
            y_train_log = np.log1p(y_train)
            y_val_log   = np.log1p(y_val)  # not strictly needed for metrics, but kept for clarity

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # log missing values BEFORE imputations
            log(f"[C{config_id}|F{fold}] NaNs before (just numerical):")
            log(str(X_train[numeric_features_fold].isna().sum()))
            log(f"[C{config_id}|F{fold}] NaNs before (categorical):")
            log(str(X_train[categorical_features].isna().sum()))
            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' before (categorical):")
            log(str(unknown_counts_before))

            # ---------------------------------
            # 2) numerical preprocessing
            # ---------------------------------

            # 2.1) YEAR
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2) MILEAGE
            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3) ENGINE SIZE
            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # 2.5) MPG
            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            # ---------------------------------
            # 2.8) BRAND
            # ---------------------------------
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )
            X_train, brand_corr_train, brand_still_invalid_train = transform_ambiguous_brands(
                X_train,
                brand_state,
            )
            X_val, brand_corr_val, brand_still_invalid_val = transform_ambiguous_brands(
                X_val,
                brand_state,
            )

            # ---------------------------------
            # 2.9) MODEL
            # ---------------------------------
            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )
            X_train, model_corr_train, model_still_invalid_train = transform_invalid_models(
                X_train,
                model_state,
            )
            X_val, model_corr_val, model_still_invalid_val = transform_invalid_models(
                X_val,
                model_state,
            )

            # ---------------------------------
            # 2.10) TRANSMISSION
            # ---------------------------------
            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )
            X_train, transm_corr_train, transm_still_invalid_train = transform_transmission_resolver(
                X_train,
                transm_state,
            )
            X_val, transm_corr_val, transm_still_invalid_val = transform_transmission_resolver(
                X_val,
                transm_state,
            )

            # ---------------------------------
            # 2.11) FUEL TYPE
            # ---------------------------------
            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )
            X_train, fuel_corr_train, fuel_still_invalid_train = transform_fueltype_resolver(
                X_train,
                fuel_state,
            )
            X_val, fuel_corr_val, fuel_still_invalid_val = transform_fueltype_resolver(
                X_val,
                fuel_state,
            )
            log(f"[C{config_id}|F{fold}] Após resolver fuelType:")
            log("  Valores distintos (train): " +
                str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))
            log("  Ainda problemáticos (train): " + str(fuel_still_invalid_train[:10]))

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            # ---------------------------------
            # Some extra logs for checking if the preprocessing went well
            # ---------------------------------
            log(f"[C{config_id}|F{fold}] --- Summary ---")
            num_means_train = X_train[numeric_features_fold].mean()
            num_nans_train  = X_train[numeric_features_fold].isna().sum()
            log("  numerical means (train):")
            log(str(num_means_train))
            log("  numerical NaNs  (train):")
            log(str(num_nans_train))

            num_means_val = X_val[numeric_features_fold].mean()
            num_nans_val  = X_val[numeric_features_fold].isna().sum()
            log("  numerical means  (val):")
            log(str(num_means_val))
            log("  numerical NaNs   (val):")
            log(str(num_nans_val))

            log(f"[C{config_id}|F{fold}] --- categorical summary ---")
            cat_nans_train     = X_train[categorical_features].isna().sum()
            cat_unknown_train  = (X_train[categorical_features] == "UNKNOWN").sum()
            log("  NaNs per categorical (train):")
            log(str(cat_nans_train))
            log("  'UNKNOWN' per categorical  (train):")
            log(str(cat_unknown_train))

            log(f"[C{config_id}|F{fold}] distinct values in categorical (train):")
            for col in categorical_features:
                uniques = sorted(X_train[col].dropna().astype(str).unique())
                if len(uniques) > 30:
                    shown = uniques[:30]
                    log(f"    {col}: {len(uniques)} distinct values. First 30: {shown} ...")
                else:
                    log(f"    {col}: {len(uniques)} distinct values: {uniques}")

            unknown_mask_train = (X_train[categorical_features] == "UNKNOWN")
            unknown_mask_val   = (X_val[categorical_features]   == "UNKNOWN")

            unknown_counts_train = unknown_mask_train.sum()
            unknown_counts_val   = unknown_mask_val.sum()

            rows_with_unknown_train = unknown_mask_train.any(axis=1).sum()
            rows_with_unknown_val   = unknown_mask_val.any(axis=1).sum()

            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after (categorical, train):")
            log(str(unknown_counts_train))
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after  (categorical, val):")
            log(str(unknown_counts_val))
            log(f"[C{config_id}|F{fold}] Num of rows with >=1 UNKNOWN (train): {rows_with_unknown_train}")
            log(f"[C{config_id}|F{fold}] Num of rows with >=1 UNKNOWN (val)  : {rows_with_unknown_val}")

            # ---------------------------------
            # 3) categorical encoding
            # ---------------------------------
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            log(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")

            # 3.1) Target Encoding para Brand e model (FIT WITH LOG TARGET)
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train_log)

            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-Hot Encoding para as restantes categóricas
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])

            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")

            unknown_cols_train = [col for col in X_train_cat.columns if "UNKNOWN" in str(col)]
            unknown_cols_val   = [col for col in X_val_cat.columns if "UNKNOWN" in str(col)]

            # ---------------------------------
            # 4) NUMERICAL SCALING
            # ---------------------------------
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features_fold])
            X_val_num   = scaler.transform(X_val[numeric_features_fold])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features_fold]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features_fold]
            )

            log(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # ---------------------------------
            # 5) concatenate scaled numeric and encoded categorical features
            # ---------------------------------
            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df,   X_val_cat],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # ---------------------------------
            # 6) instantiate and train the random forest regressor for this configuration
            # ---------------------------------
            rf = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **params
            )

            log(f"[C{config_id}|F{fold}] training RandomForestRegressor (log-target)...")
            rf.fit(X_train_final, y_train_log)

            # predictions in log-space -> back-transform to euros for evaluation
            y_pred_train_log = rf.predict(X_train_final)
            y_pred_val_log   = rf.predict(X_val_final)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)

            # ---------------------------------
            # 7) compute metrics per fold for both train and validation (EUROS)
            # ---------------------------------
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)

            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)

            log(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.4f} | MAE: {mae_tr:.4f} | R²: {r2_tr:.4f}")
            log(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R²: {r2_val:.4f}")

            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)

            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)

        # average metrics of all folds for this configuration
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)

        mean_rmse_tr = np.mean(fold_rmses_train)
        mean_mae_tr  = np.mean(fold_maes_train)
        mean_r2_tr   = np.mean(fold_r2s_train)

        # combined score giving equal weight to rmse and mae on validation
        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - TRAIN RMSE mean: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f} | R²: {mean_r2_tr:.4f}")
        log(f"Config {config_id} - VAL   RMSE mean: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | R²: {mean_r2_val:.4f}")
        log(f"Config {config_id} - Score combined (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "combo_score": combo_score,
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id} w/ mean RMSE (VAL) = {best_rmse:.4f}")

        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id} w/ mean MAE  (VAL) = {best_mae:.4f}")

        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED] Config {config_id} w/ score = {best_combo:.4f}")

    log("")
    log("")
    log("# =============================")
    log("# END")
    log("# =============================")
    log(f"Best config (min RMSE VAL): {best_config}")
    log(f"best mean RMSE  (VAL): {best_rmse:.4f}")
    log(f"Best config (min MAE VAL): {best_config_mae}")
    log(f"BEST mean MAE (VAL): {best_mae:.4f}")
    log(f"best config (combined score VAL): {best_config_combo}")
    log(f"best combined score (VAL): {best_combo:.4f}")

# -----------------------------
# final summary of the random search (as a dataframe)
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True)

display(results_df_sorted.head(10))

print("\nBest config (min RMSE VAL):")
print(best_config)
print("best mean RMSE (VAL):", best_rmse)

print("\nbest config (min MAE VAL):")
print(best_config_mae)
print("best mean MAE  (VAL):", best_mae)

print("\nbest config found (combined score VAL = 0.5*RMSE + 0.5*MAE):")
print(best_config_combo)
print("best combined score (VAL):", best_combo)

print(f"\nLogs stored at: {log_path}")


# starting random forest
N_SPLITS = 8, N_RANDOM_CONFIGS = 15
param_distributions = {'n_estimators': [200, 400, 600, 800, 1000], 'max_depth': [7, 9, 11, 15, 20], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 4, 6, 8], 'max_features': ['sqrt', 0.33, 0.5, 1.0], 'bootstrap': [True]}
Target transform: log1p(price) during training; metrics computed back in euros via expm1(pred).
Note: previousOwners is not used (excluded from numeric_features and dropped per fold).

 - CONFIG 1/15 
Parameters: {'n_estimators': 400, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 11, 'bootstrap': True}

[CONFIG 1] -- FOLD 1/8
[C1|F1] X_train shape: (66476, 9), X_val shape: (9497, 9)
[C1|F1] y_train shape: (66476,), y_val shape: (9497,)
[C1|F1] NaNs before (just numerical):
year          1306
mileage       1252
engineSize    1344
tax           6940
mpg           6961
dtype: int64
[C1|F1] NaNs before (categorical):
Brand           0
model           0
transmissio

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_train_mean,mae_train_mean,r2_train_mean,rmse_mean,mae_mean,r2_mean,combo_score
6,7,1000,2,2,0.33,20,True,1749.613634,966.927495,0.967707,2376.031570,1306.843493,0.940243,1841.437531
1,2,1000,10,4,0.5,15,True,2061.164838,1162.433703,0.955184,2414.308057,1354.123910,0.938320,1884.215984
8,9,800,10,2,sqrt,20,True,2090.574201,1129.708403,0.953897,2535.587984,1366.144897,0.931972,1950.866440
13,14,600,2,8,0.5,15,True,2293.772583,1260.733852,0.944499,2546.586228,1402.361378,0.931411,1974.473803
11,12,800,10,2,sqrt,15,True,2247.398643,1266.204570,0.946722,2606.574134,1433.685997,0.928141,2020.130065
7,8,1000,2,4,0.5,11,True,2403.005051,1423.505121,0.939089,2617.614765,1518.454457,0.927565,2068.034611
0,1,400,4,2,1.0,11,True,2317.639853,1424.902550,0.943340,2619.219163,1553.037194,0.927478,2086.128179
4,5,200,6,2,1.0,11,True,2341.113557,1429.937417,0.942187,2627.828991,1554.615266,0.926999,2091.222128
5,6,200,10,8,0.5,11,True,2533.152801,1458.113861,0.932313,2703.905019,1539.446499,0.922719,2121.675759
3,4,1000,6,8,sqrt,15,True,2699.447436,1442.454293,0.923135,2884.552549,1540.446587,0.912015,2212.499568



Best config (min RMSE VAL):
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True}
best mean RMSE (VAL): 2376.03157003845

best config (min MAE VAL):
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True}
best mean MAE  (VAL): 1306.8434926812392

best config found (combined score VAL = 0.5*RMSE + 0.5*MAE):
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True}
best combined score (VAL): 1841.4375313598446

Logs stored at: rf_random_search_log_2_.txt


### 3.3. Feature Engineering Ablations and Additions <a id="feateng"></a>

##### 3.3.3.1. Without Feature Selection <a id="333nofs"></a>

In [ ]:
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

# set core configuration for cross validation
N_SPLITS = 8 # number of folds used in k-fold cross validation
RANDOM_STATE = 42

# creation of the KFold object that will generate the train/validation splits
kf = KFold(
    n_splits=N_SPLITS, # number of folds, in this case will be 8 folds
    shuffle=True,  # shuffle data before splitting to make folds more balanced
    random_state=RANDOM_STATE # using the fixed seed 
)

# ------ // ------
# Definition of the hyperparameters that will be tested in the Randomized Search

param_distributions = {
    # number of trees in the forest: 
    "n_estimators": [200, 400, 600, 800, 1000], 
    # maximum depth of each tree:
    "max_depth": [5, 7, 9, 11, 15, 20],
    # minimum number of samples required to split an internal node:
    "min_samples_split": [2, 4, 6, 10],
    # minimum number of samples required to be at a leaf node:
    "min_samples_leaf": [1, 2, 4, 8],
    # number of features to consider when looking for the best split:
    "max_features": [None], # using all features
    "bootstrap": [True, False], # whether bootstrap samples are used when building trees
}

N_RANDOM_CONFIGS = 10 # number of random different hyperparameter configurations to try


# this sampler will generate random combinations of hyperparameters
# from the previously defined search space 
sampler = ParameterSampler(
    param_distributions, # the dictionary with the hyperparameter search space
    n_iter=N_RANDOM_CONFIGS, # number of different random configurations to generate
    random_state=RANDOM_STATE 
)

search_results = [] # list that will store summary results for each hyperparameter configuration

# best configuration according to validation rmse
best_rmse = np.inf # our initial best rmse is infinite so any real rmse will be smaller
best_config = None # will store the params corresponding to the best rmse

# best configuration according to validation mae
best_mae = np.inf
best_config_mae = None

# best configuration according to a combined metric 0.5 * rmse + 0.5 * mae on validation
best_combo = np.inf
best_config_combo = None

# we have a log file to store detailed logs of the random search process
# this is useful for later analysis of the results, to avoid losing information if the notebook crashes
# and following the progress of the search
log_path = "rf_random_search_log__4_.txt"

# opens the log file for writing
with open(log_path, "w", encoding="utf-8") as log_file:

    # this is a helper function to write messages to both console and log file
    def log(msg: str):
        print(msg) # print to stdout so progress is visible during execution
        log_file.write(msg + "\n") # also write the same message to the log file
        log_file.flush() # flush so log is updated immediately on disk

    log("# starting random forest")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")

    # -----------------------------
    # main loop of the random search with cross validation
    # each iteration uses one sampled hyperparameter configuration
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f" - CONFIG {config_id}/{N_RANDOM_CONFIGS} ")
        log(f"Parameters: {params}")

        # lists to store validation metrics per fold for this configuration
        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        # lists to store training metrics per fold for this configuration
        fold_rmses_train = []
        fold_maes_train  = []
        fold_r2s_train   = []

        # cross validation loop for the current hyperparameter configuration
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] -- FOLD {fold}/{N_SPLITS}")

            # ---------------------------------
            # 1) split the data into train and validation for the current fold
            # ---------------------------------
            X_train = X.iloc[train_idx].copy() # training features for the current fold
            X_val   = X.iloc[val_idx].copy() # validation features for the current fold
            y_train = y.iloc[train_idx].copy() # training targets for the current fold
            y_val   = y.iloc[val_idx].copy() # validation targets for the current fold

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # log missing values BEFORE imputations
            # --> for the numeric features
            log(f"[C{config_id}|F{fold}] NaNs before (just numerical):")
            log(str(X_train[numeric_features].isna().sum()))
            # --> for the categorical features (we are not supposed to have NaNs here because they were already temporarily handled as UNKNOWN, but just in case)
            log(f"[C{config_id}|F{fold}] NaNs before (categorical):")
            log(str(X_train[categorical_features].isna().sum()))
            # count how many "UNKNOWN" exist in the categorical features before further processing
            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' before (categorical):")
            log(str(unknown_counts_before))

            # ---------------------------------
            # 2) numerical preprocessing 
            #    using fit on train and transform on val
            #    each "state" object stores statistics or rules learned from train
            # ---------------------------------

            # 2.1) YEAR 
            # impute year based on median year per model learned from training data
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2) MILEAGE
            # impute mileage using a custom imputer that can also enforce absolute values
            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3) ENGINE SIZE
            # impute engine size using a imputer fitted on the training data
            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)


            # 2.5) MPG
            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            
            # 2.7) PREVIOUS OWNERS
            owners_state = fit_previous_owners_imputer(
                X_train,
                owners_col="previousOwners",
                year_col="year",
                mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            

            # ---------------------------------
            # 2.8) BRAND 
            # ---------------------------------
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )
            X_train, brand_corr_train, brand_still_invalid_train = transform_ambiguous_brands(
                X_train,
                brand_state,
            )
            X_val, brand_corr_val, brand_still_invalid_val = transform_ambiguous_brands(
                X_val,
                brand_state,
            )
            

            # ---------------------------------
            # 2.9) MODEL
            # ---------------------------------
            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )
            X_train, model_corr_train, model_still_invalid_train = transform_invalid_models(
                X_train,
                model_state,
            )
            X_val, model_corr_val, model_still_invalid_val = transform_invalid_models(
                X_val,
                model_state,
            )
             

            # ---------------------------------
            # 2.10) TRANSMISSION
            # ---------------------------------
            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )
            X_train, transm_corr_train, transm_still_invalid_train = transform_transmission_resolver(
                X_train,
                transm_state,
            )
            X_val, transm_corr_val, transm_still_invalid_val = transform_transmission_resolver(
                X_val,
                transm_state,
            )
            
            
            # ---------------------------------
            # 2.11) FUEL TYPE
            # ---------------------------------
            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )
            X_train, fuel_corr_train, fuel_still_invalid_train = transform_fueltype_resolver(
                X_train,
                fuel_state,
            )
            X_val, fuel_corr_val, fuel_still_invalid_val = transform_fueltype_resolver(
                X_val,
                fuel_state,
            )
            # --
            log(f"[C{config_id}|F{fold}] Após resolver fuelType:")
            log("  Valores distintos (train): " +
                str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))
            log("  Ainda problemáticos (train): " + str(fuel_still_invalid_train[:10]))
            # --

            # ------------------------------------------------------------------
            # (after fuel resolver + tax rules) -> FEATURE ENGINEERING (FE)
            # ------------------------------------------------------------------

            # existing tax rule transform (keep as you have it)
            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

            # -> Feature Engineering
            # replace previousOwners with a more robust derived feature (and drop original)
            X_train = add_owners_flagged(X_train, owners_col="previousOwners",
                                        new_col="owners_flagged", drop_original=True)
            X_val   = add_owners_flagged(X_val,   owners_col="previousOwners",
                                        new_col="owners_flagged", drop_original=True)

            # age feature: uses a fixed reference year (base_year=2020) and drops "year"
            X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
            X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

            # mileage-derived features; drops original mileage (and drops ratio feature if created)
            X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age",
                                        drop_original=True, drop_ratio=True)
            X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age",
                                        drop_original=True, drop_ratio=True)

            # discretize engineSize into bins; treat as categorical (string) for OHE
            X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
            X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

            X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
            X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

            # ------------------------------------------------------------------
            # 3) CATEGORICAL ENCODING (TargetEnc for high-card, OHE for low-card)
            # ------------------------------------------------------------------

            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            # update low-card set to include engineered categorical bin
            low_card_curr = low_card_features + ["engine_bin"]

            # 3.1) Target Encoding for Brand + model
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)         # NOTE: y_train is in euros in your RF pipeline
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-Hot Encoding for the remaining categoricals (+ engine_bin)
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_curr])
            X_train_low = ohe.transform(X_train[low_card_curr])
            X_val_low   = ohe.transform(X_val[low_card_curr])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # ------------------------------------------------------------------
            # 4) NUMERICAL "SCALING" (keep your approach, but recompute numeric cols)
            # ------------------------------------------------------------------
            # IMPORTANT: after FE you dropped 'year', 'mileage', 'previousOwners', so we must
            # scale the numeric columns that actually exist now (otherwise KeyError).

            cat_cols_curr = set(high_card_features + low_card_curr)
            numeric_features_curr = [c for c in X_train.columns if c not in cat_cols_curr]

            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features_curr])
            X_val_num   = scaler.transform(X_val[numeric_features_curr])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features_curr],
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features_curr],
            )

            # ------------------------------------------------------------------
            # 5) CONCAT numeric + categorical
            # ------------------------------------------------------------------
            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df,   X_val_cat],   axis=1)

            # Safety: ensure exact same columns/order in validation (missing dummies -> 0)
            X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

            # ---------------------------------
            # 6) instantiate and train the random forest regressor for this configuration
            # ---------------------------------
            rf = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1, # uses all available cores 
                **params # current hyperparameter configuration
            )

            log(f"[C{config_id}|F{fold}] training RandomForestRegressor...")
            rf.fit(X_train_final, y_train) # train the model on the fully processed training data

            # generate predictions on training and validation sets for metric computation
            y_pred_train = rf.predict(X_train_final)
            y_pred_val   = rf.predict(X_val_final)

            # ---------------------------------
            # 7) compute metrics per fold for both train and validation
            #    used to monitor performance and detect potential overfitting/underfitting
            # ---------------------------------
            # Validation metrics:
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)

            # TRAIN metrics: 
            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)

            log(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.4f} | MAE: {mae_tr:.4f} | R²: {r2_tr:.4f}")
            log(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R²: {r2_val:.4f}")

            # store the validation metrics for this fold
            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)

            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)

        # average metrics of all folds for this configuration
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)

        mean_rmse_tr = np.mean(fold_rmses_train)
        mean_mae_tr  = np.mean(fold_maes_train)
        mean_r2_tr   = np.mean(fold_r2s_train)

        # combined score giving equal weight to rmse and mae on validation
        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - TRAIN RMSE mean: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f} | R²: {mean_r2_tr:.4f}")
        log(f"Config {config_id} - VAL   RMSE mean: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | R²: {mean_r2_val:.4f}")
        log(f"Config {config_id} - Score combined (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.4f}")


        # store a record with configuration, metrics and combined score to the search results list
        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "combo_score": combo_score,
        })

        # update (or not) RMSE (VAL)
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id} w/ mean RMSE (VAL) = {best_rmse:.4f}")

        # update (or not) MAE (VAL)
        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id} w/ mean MAE  (VAL) = {best_mae:.4f}")

        # update (or not) best combined score (VAL)
        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED] Config {config_id} w/ score = {best_combo:.4f}")


    # after iterating over all sampled configurations, log a summary of the best ones
    log("")
    log("")
    log("# =============================")
    log("# END")
    log("# =============================")
    log(f"Best config (min RMSE VAL): {best_config}")
    log(f"best mean RMSE  (VAL): {best_rmse:.4f}")
    log(f"Best config (min MAE VAL): {best_config_mae}")
    log(f"BEST mean MAE (VAL): {best_mae:.4f}")
    log(f"best config (combined score VAL): {best_config_combo}")
    log(f"best combined score (VAL): {best_combo:.4f}")

# -----------------------------
# final summary of the random search (as a dataframe)
# -----------------------------
results_df = pd.DataFrame(search_results) # convert list of dicts to dataframe
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True) # sort by validation rmse

display(results_df_sorted.head(10)) # configurations with lowest validation rmse

# print the best configurations according to each criterion
print("\nBest config (min RMSE VAL):")
print(best_config)
print("best mean RMSE (VAL):", best_rmse)

print("\nbest config (min MAE VAL):")
print(best_config_mae)
print("best mean MAE  (VAL):", best_mae)

print("\nbest config found (combined score VAL = 0.5*RMSE + 0.5*MAE):")
print(best_config_combo)
print("best combined score (VAL):", best_combo)

print(f"\nLogs stored at: {log_path}")


# starting random forest
N_SPLITS = 8, N_RANDOM_CONFIGS = 10
param_distributions = {'n_estimators': [200, 400, 600, 800, 1000], 'max_depth': [5, 7, 9, 11, 15, 20], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 4, 8], 'max_features': [None], 'bootstrap': [True, False]}

 - CONFIG 1/10 
Parameters: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 15, 'bootstrap': False}

[CONFIG 1] -- FOLD 1/8
[C1|F1] X_train shape: (66476, 10), X_val shape: (9497, 10)
[C1|F1] y_train shape: (66476,), y_val shape: (9497,)
[C1|F1] NaNs before (just numerical):
year              1306
mileage           1252
engineSize        1344
tax               6940
mpg               6961
previousOwners    1331
dtype: int64
[C1|F1] NaNs before (categorical):
Brand           0
model           0
transmission    0
fuelType        0
dtype: int64
[C1|F1] 'UNKNOWN' before (categorical):
Brand           1328
model           1314
transmission    1944
fuelTy

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_train_mean,mae_train_mean,r2_train_mean,rmse_mean,mae_mean,r2_mean,combo_score
7,8,1000,6,1,None,15,True,1555.556593,999.456009,0.974475,2254.876016,1332.693673,0.946205,1793.784845
2,3,200,6,2,None,15,True,1649.808664,1016.060652,0.971286,2295.227361,1336.887394,0.944252,1816.057378
1,2,600,10,8,None,20,True,2063.138372,1147.497713,0.955098,2446.120598,1379.233152,0.936681,1912.676875
5,6,400,2,2,None,11,True,2097.090435,1358.879903,0.953610,2472.656697,1507.231327,0.935364,1989.944012
0,1,400,10,2,None,15,False,1825.338331,1130.773975,0.964840,2672.540166,1536.944166,0.924468,2104.742166
3,4,800,6,1,None,20,False,1185.878034,686.438196,0.985154,2725.719578,1562.094808,0.921492,2143.907193
9,10,800,10,8,None,15,False,2195.442902,1278.639216,0.949155,2733.267688,1554.137577,0.920812,2143.702632
4,5,800,2,4,None,20,False,1729.879072,977.201043,0.968430,2735.190561,1546.211872,0.920903,2140.701217
6,7,200,10,4,None,9,True,2547.400316,1621.831141,0.931551,2749.840174,1696.666699,0.920112,2223.253436
8,9,400,4,1,None,7,True,2996.441253,1946.018791,0.905292,3125.900761,1985.090371,0.896835,2555.495566



Best config (min RMSE VAL):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
best mean RMSE (VAL): 2254.8760157737897

best config (min MAE VAL):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
best mean MAE  (VAL): 1332.6936732652434

best config found (combined score VAL = 0.5*RMSE + 0.5*MAE):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
best combined score (VAL): 1793.7848445195166

Logs stored at: rf_random_search_log__4_.txt


##### 3.3.3.2. With Feature Selection - Randomized Search <a id="agenoown80"></a>

In [ ]:
# ==============================================================================
# RANDOM SEARCH (RF) + FEATURE ENGINEERING + FEATURE SELECTION (SelectFromModel)
# + NUMERICAL SCALING (StandardScaler)
# ==============================================================================

import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------------
# 0) CONFIG
# -----------------------------
RANDOM_STATE = 42
N_SPLITS = 8
N_ITER = 15

# Feature-selection ratios to be searched
FS_KEEP_RATIOS = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

# RF used ONLY for feature selection (kept fixed to make selection stable)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# Reproducibility for the sampling of configs
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# -----------------------------
# Helpers (metrics)
# -----------------------------
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def bias(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

# -----------------------------
# 1) CV splitter
# -----------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# -----------------------------
# 2) Feature lists
# -----------------------------
numeric_features = ["year", "mileage", "engineSize", "tax", "mpg", "previousOwners"]
categorical_features = ["Brand", "model", "transmission", "fuelType"]

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize categorical strings (reduces spurious categories).
    Assumes fill_unknown + column_string_transformer exist.
    """
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(df, column=col, remove_middle_spaces=True, allow_extra_chars="")
    return df

# -----------------------------
# 3) Random Search space (RF model hyperparams + fs_keep_ratio)
# -----------------------------
param_distributions = {
    # --- RF hyperparameters (main model) ---
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [7, 9, 11, 15, 20],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "max_features": ["sqrt", 0.33, 0.5, 1.0],
    "bootstrap": [True],

    # --- NEW: FS ratio ---
    "fs_keep_ratio": FS_KEEP_RATIOS,
}

# Sample N_ITER configs (deterministic due to seeds above)
sampler = list(ParameterSampler(param_distributions, n_iter=N_ITER, random_state=RANDOM_STATE))
CONFIGS = [(f"CFG_{i+1:03d}", cfg) for i, cfg in enumerate(sampler)]

# -----------------------------
# 4) Evaluate one config (8-fold CV)
# -----------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        # Split fold data
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # Normalize categoricals early
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Keep only expected base columns
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING / IMPUTATION (fit on TRAIN fold only)
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS (fit on TRAIN fold only)
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # tax rules depend on year/fuelType/engineSize -> keep BEFORE dropping year
        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners",
                                     new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners",
                                     new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age",
                                       drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age",
                                       drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = list(dict.fromkeys(low_card_features + ["engine_bin"]))

        # -----------------------------------------------------
        # D) ENCODING (TE for high-card, OHE for low-card)
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

        # Numeric columns are everything except the categorical inputs
        cat_cols_curr = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in cat_cols_curr]

        # ---------------------------------
        # 4) NUMERICAL SCALING  (your block, but using numeric_features_curr)
        # ---------------------------------
        scaler = StandardScaler()
        X_train_num = scaler.fit_transform(X_train[numeric_features_curr])  # fit on train, transform train
        X_val_num   = scaler.transform(X_val[numeric_features_curr])        # transform val with same scaler

        # convert arrays back to dataframes (stable indices + explicit names)
        X_train_num_df = pd.DataFrame(
            X_train_num,
            index=X_train.index,
            columns=[f"num_{col}" for col in numeric_features_curr]
        )
        X_val_num_df = pd.DataFrame(
            X_val_num,
            index=X_val.index,
            columns=[f"num_{col}" for col in numeric_features_curr]
        )

        # Combine numeric + categorical
        X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val_num_df,   X_val_cat], axis=1)

        # Align validation columns with train (missing dummies -> 0)
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # E) FEATURE SELECTION (SelectFromModel + RF)  [per-fold]
        # -----------------------------------------------------
        fs_keep_ratio = params["fs_keep_ratio"]

        n_total = X_train_final.shape[1]
        k = int(np.ceil(fs_keep_ratio * n_total))
        k = max(1, min(k, n_total))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        selected_cols = X_train_final.columns[selector.get_support()]
        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # F) Train main RF on selected features
        # -----------------------------------------------------
        rf_params = {k: v for k, v in params.items() if k != "fs_keep_ratio"}

        rf = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **rf_params
        )
        rf.fit(X_train_sel, y_train)

        pred_tr  = rf.predict(X_train_sel)
        pred_val = rf.predict(X_val_sel)

        fold_rows.append({
            "config": name,
            "fold": fold,

            "fs_keep_ratio": fs_keep_ratio,
            "n_features_total": int(n_total),
            "n_features_selected": int(X_train_sel.shape[1]),

            "train_rmse": rmse(y_train, pred_tr),
            "val_rmse":   rmse(y_val,   pred_val),

            "train_mae": float(mean_absolute_error(y_train, pred_tr)),
            "val_mae":   float(mean_absolute_error(y_val,   pred_val)),

            "train_r2": float(r2_score(y_train, pred_tr)),
            "val_r2":   float(r2_score(y_val,   pred_val)),

            "train_bias": bias(y_train, pred_tr),
            "val_bias":   bias(y_val,   pred_val),
        })

    df_folds = pd.DataFrame(fold_rows)

    out = {
        "config": name,

        "val_rmse_mean": float(df_folds["val_rmse"].mean()),
        "val_mae_mean":  float(df_folds["val_mae"].mean()),
        "val_r2_mean":   float(df_folds["val_r2"].mean()),
        "val_bias_mean": float(df_folds["val_bias"].mean()),

        "train_rmse_mean": float(df_folds["train_rmse"].mean()),
        "train_mae_mean":  float(df_folds["train_mae"].mean()),
        "train_r2_mean":   float(df_folds["train_r2"].mean()),
        "train_bias_mean": float(df_folds["train_bias"].mean()),

        "avg_total_features": float(df_folds["n_features_total"].mean()),
        "avg_selected_features": float(df_folds["n_features_selected"].mean()),

        **params,
    }

    print(
        f"   >> [{name}] "
        f"VAL RMSE: {out['val_rmse_mean']:.1f} | VAL MAE: {out['val_mae_mean']:.1f} | "
        f"KeepRatio: {out['fs_keep_ratio']:.2f} | Sel: {out['avg_selected_features']:.0f}/{out['avg_total_features']:.0f}"
    )

    return out

# -----------------------------
# 5) RUN RANDOM SEARCH
# -----------------------------
results = []
best_rmse = np.inf
best_row = None

for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

    if res["val_rmse_mean"] < best_rmse:
        best_rmse = res["val_rmse_mean"]
        best_row = res

results_df = pd.DataFrame(results)

print("\n- TOP 10 Configurations (sorted by VAL RMSE)")
results_df_sorted = results_df.sort_values("val_rmse_mean", ascending=True)

cols_show = [
    "config",
    "val_rmse_mean", "val_mae_mean", "val_r2_mean", "val_bias_mean",
    "train_rmse_mean", "train_mae_mean", "train_r2_mean", "train_bias_mean",
    "fs_keep_ratio", "avg_selected_features", "avg_total_features",
    "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features", "bootstrap",
]
display(results_df_sorted[cols_show].head(10))

print("\n- BEST CONFIG (by VAL RMSE)")
print(f"Best config: {best_row['config']}")
print(f"VAL RMSE: {best_row['val_rmse_mean']:.3f}")
print(f"VAL MAE:  {best_row['val_mae_mean']:.3f}")
print(f"VAL R2:   {best_row['val_r2_mean']:.6f}")
print(f"Keep ratio: {best_row['fs_keep_ratio']:.2f}")
print(f"Avg selected feats: {best_row['avg_selected_features']:.0f}/{best_row['avg_total_features']:.0f}")

best_params = {k: best_row[k] for k in param_distributions.keys() if k in best_row}
print("Best params dict:")
print(best_params)



-- doing 1/15
   >> [CFG_001] VAL RMSE: 2544.0 | VAL MAE: 1468.6 | KeepRatio: 0.70 | Sel: 18/25
   >> [CFG_002] VAL RMSE: 2739.6 | VAL MAE: 1647.3 | KeepRatio: 0.55 | Sel: 14/25
   >> [CFG_003] VAL RMSE: 2536.1 | VAL MAE: 1452.8 | KeepRatio: 0.65 | Sel: 17/25
   >> [CFG_004] VAL RMSE: 2590.0 | VAL MAE: 1532.2 | KeepRatio: 0.65 | Sel: 17/25
   >> [CFG_005] VAL RMSE: 2395.3 | VAL MAE: 1351.8 | KeepRatio: 0.80 | Sel: 20/25

-- doing 6/15
   >> [CFG_006] VAL RMSE: 2648.5 | VAL MAE: 1518.7 | KeepRatio: 0.80 | Sel: 20/25
   >> [CFG_007] VAL RMSE: 2273.2 | VAL MAE: 1317.3 | KeepRatio: 0.65 | Sel: 17/25
   >> [CFG_008] VAL RMSE: 2995.2 | VAL MAE: 1867.7 | KeepRatio: 0.70 | Sel: 18/25
   >> [CFG_009] VAL RMSE: 2934.3 | VAL MAE: 1807.9 | KeepRatio: 0.55 | Sel: 14/25
   >> [CFG_010] VAL RMSE: 2821.3 | VAL MAE: 1759.7 | KeepRatio: 0.65 | Sel: 17/25

-- doing 11/15
   >> [CFG_011] VAL RMSE: 2414.4 | VAL MAE: 1374.5 | KeepRatio: 0.65 | Sel: 17/25
   >> [CFG_012] VAL RMSE: 2383.3 | VAL MAE: 1372.0 |

,config,val_rmse_mean,val_mae_mean,val_r2_mean,val_bias_mean,train_rmse_mean,train_mae_mean,train_r2_mean,train_bias_mean,fs_keep_ratio,avg_selected_features,avg_total_features,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,bootstrap
6,CFG_007,2273.225300,1317.265943,0.945339,6.813836,1596.479566,942.001071,0.973114,1.429380,0.65,17.0,25.0,1000,20,6,2,0.33,True
11,CFG_012,2383.288879,1371.975845,0.939934,5.939312,1868.676689,1096.534835,0.963165,1.155778,0.80,20.0,25.0,400,20,10,2,sqrt,True
4,CFG_005,2395.332345,1351.794850,0.939252,5.933997,1930.543046,1075.649194,0.960685,1.044446,0.80,20.0,25.0,1000,20,2,6,1.0,True
10,CFG_011,2414.422432,1374.513032,0.938294,7.151204,2008.816530,1154.002352,0.957433,1.943112,0.65,17.0,25.0,800,15,10,6,1.0,True
13,CFG_014,2452.780056,1510.275461,0.936422,7.448438,2081.396379,1364.939460,0.954302,3.434210,0.80,20.0,25.0,400,11,6,1,1.0,True
12,CFG_013,2466.136529,1420.986606,0.935697,6.925135,2155.995262,1257.595729,0.950968,2.485656,0.80,20.0,25.0,800,15,2,6,0.33,True
2,CFG_003,2536.072634,1452.827584,0.932016,6.457582,2270.228057,1310.908797,0.945635,2.421299,0.65,17.0,25.0,200,15,6,8,0.33,True
0,CFG_001,2543.973203,1468.645637,0.931592,7.148921,2261.214822,1322.171384,0.946065,2.655537,0.70,18.0,25.0,200,15,6,6,sqrt,True
3,CFG_004,2590.027353,1532.185287,0.929068,7.728277,2353.206470,1421.367943,0.941587,3.158334,0.65,17.0,25.0,400,11,6,8,1.0,True
5,CFG_006,2648.525875,1518.663760,0.925857,6.248184,2419.841108,1400.133868,0.938234,2.216897,0.80,20.0,25.0,1000,15,2,8,sqrt,True



- BEST CONFIG (by VAL RMSE)
Best config: CFG_007
VAL RMSE: 2273.225
VAL MAE:  1317.266
VAL R2:   0.945339
Keep ratio: 0.65
Avg selected feats: 17/25
Best params dict:
{'n_estimators': 1000, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 0.33, 'bootstrap': True, 'fs_keep_ratio': 0.65}


#### 3.3.3. Combining `age` and Excluding `previousOwners` Features

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, ParameterSampler
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------------------------------------------------
# 0) FEATURES
# ---------------------------------------------------------
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg']  # we removed previousOwners feature from here
categorical_features = ["Brand", "model", "transmission", "fuelType"]

# IMPORTANT: keep a baseline copy to avoid leaking state between folds/configs
BASE_NUMERIC_FEATURES = numeric_features.copy()

# set core configuration for cross validation
N_SPLITS = 8  # number of folds used in k-fold cross validation
RANDOM_STATE = 42

# creation of the KFold object that will generate the train/validation splits
kf = KFold(
    n_splits=N_SPLITS,  # number of folds, in this case will be 8 folds
    shuffle=True,  # shuffle data before splitting to make folds more balanced
    random_state=RANDOM_STATE  # using the fixed seed
)

# ------ // ------
# Definition of the hyperparameters that will be tested in the Randomized Search
param_distributions = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [7, 9, 11, 15, 20],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "max_features": ["sqrt", 0.33, 0.5, 1.0],
    "bootstrap": [True],
}

N_RANDOM_CONFIGS = 15  # number of random different hyperparameter configurations to try

# this sampler will generate random combinations of hyperparameters
# from the previously defined search space
sampler = ParameterSampler(
    param_distributions,  # the dictionary with the hyperparameter search space
    n_iter=N_RANDOM_CONFIGS,  # number of different random configurations to generate
    random_state=RANDOM_STATE
)

search_results = []  # list that will store summary results for each hyperparameter configuration

# best configuration according to validation rmse
best_rmse = np.inf  # our initial best rmse is infinite so any real rmse will be smaller
best_config = None  # will store the params corresponding to the best rmse

# best configuration according to validation mae
best_mae = np.inf
best_config_mae = None

# best configuration according to a combined metric 0.5 * rmse + 0.5 * mae on validation
best_combo = np.inf
best_config_combo = None

# we have a log file to store detailed logs of the random search process
log_path = "rf_random_search_log_.txt"

# opens the log file for writing
with open(log_path, "w", encoding="utf-8") as log_file:

    # this is a helper function to write messages to both console and log file
    def log(msg: str):
        print(msg)  # print to stdout so progress is visible during execution
        log_file.write(msg + "\n")  # also write the same message to the log file
        log_file.flush()  # flush so log is updated immediately on disk

    log("# starting random forest")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")
    log("Target transform: log1p(price) during training; metrics computed back in euros via expm1(pred).")
    log("Note: previousOwners is not used (excluded from numeric_features and dropped per fold).")

    # -----------------------------
    # main loop of the random search with cross validation
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f" - CONFIG {config_id}/{N_RANDOM_CONFIGS} ")
        log(f"Parameters: {params}")

        # lists to store validation metrics per fold for this configuration
        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        # lists to store training metrics per fold for this configuration
        fold_rmses_train = []
        fold_maes_train  = []
        fold_r2s_train   = []

        # cross validation loop for the current hyperparameter configuration
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] -- FOLD {fold}/{N_SPLITS}")

            # LOCAL numeric feature list for this fold (prevents state leaking between folds)
            numeric_features_fold = BASE_NUMERIC_FEATURES.copy()

            # ---------------------------------
            # 1) split the data into train and validation for the current fold
            # ---------------------------------
            X_train = X.iloc[train_idx].copy()  # training features for the current fold
            X_val   = X.iloc[val_idx].copy()    # validation features for the current fold
            y_train = y.iloc[train_idx].copy()  # training targets for the current fold (EUROS)
            y_val   = y.iloc[val_idx].copy()    # validation targets for the current fold (EUROS)

            # EXTRA GUARANTEE: drop previousOwners so it can never leak into the final matrix
            X_train = X_train.drop(columns=["previousOwners"], errors="ignore")
            X_val   = X_val.drop(columns=["previousOwners"], errors="ignore")

            # LOG TARGET (training happens in log space)
            y_train_log = np.log1p(y_train)
            y_val_log   = np.log1p(y_val)  # not strictly needed for metrics, but kept for clarity

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # log missing values BEFORE imputations
            log(f"[C{config_id}|F{fold}] NaNs before (just numerical):")
            log(str(X_train[numeric_features_fold].isna().sum()))
            log(f"[C{config_id}|F{fold}] NaNs before (categorical):")
            log(str(X_train[categorical_features].isna().sum()))
            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' before (categorical):")
            log(str(unknown_counts_before))

            # ---------------------------------
            # 2) numerical preprocessing
            # ---------------------------------

            # 2.1) YEAR
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2) MILEAGE
            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3) ENGINE SIZE
            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # 2.5) MPG
            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            # ---------------------------------
            # 2.8) BRAND
            # ---------------------------------
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )
            X_train, brand_corr_train, brand_still_invalid_train = transform_ambiguous_brands(
                X_train,
                brand_state,
            )
            X_val, brand_corr_val, brand_still_invalid_val = transform_ambiguous_brands(
                X_val,
                brand_state,
            )

            # ---------------------------------
            # 2.9) MODEL
            # ---------------------------------
            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )
            X_train, model_corr_train, model_still_invalid_train = transform_invalid_models(
                X_train,
                model_state,
            )
            X_val, model_corr_val, model_still_invalid_val = transform_invalid_models(
                X_val,
                model_state,
            )

            # ---------------------------------
            # 2.10) TRANSMISSION
            # ---------------------------------
            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )
            X_train, transm_corr_train, transm_still_invalid_train = transform_transmission_resolver(
                X_train,
                transm_state,
            )
            X_val, transm_corr_val, transm_still_invalid_val = transform_transmission_resolver(
                X_val,
                transm_state,
            )

            # ---------------------------------
            # 2.11) FUEL TYPE
            # ---------------------------------
            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )
            X_train, fuel_corr_train, fuel_still_invalid_train = transform_fueltype_resolver(
                X_train,
                fuel_state,
            )
            X_val, fuel_corr_val, fuel_still_invalid_val = transform_fueltype_resolver(
                X_val,
                fuel_state,
            )
            log(f"[C{config_id}|F{fold}] Após resolver fuelType:")
            log("  Valores distintos (train): " +
                str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))
            log("  Ainda problemáticos (train): " + str(fuel_still_invalid_train[:10]))

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            # ------------------------------------------------------------------
            # NEW: age feature (drops 'year')
            # ------------------------------------------------------------------
            X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
            X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

            # update local numeric features AFTER dropping year
            numeric_features_fold = ["age", "mileage", "engineSize", "tax", "mpg"]

            # ---------------------------------
            # Some extra logs for checking if the preprocessing went well
            # ---------------------------------
            log(f"[C{config_id}|F{fold}] --- Summary ---")
            num_means_train = X_train[numeric_features_fold].mean()
            num_nans_train  = X_train[numeric_features_fold].isna().sum()
            log("  numerical means (train):")
            log(str(num_means_train))
            log("  numerical NaNs  (train):")
            log(str(num_nans_train))

            num_means_val = X_val[numeric_features_fold].mean()
            num_nans_val  = X_val[numeric_features_fold].isna().sum()
            log("  numerical means  (val):")
            log(str(num_means_val))
            log("  numerical NaNs   (val):")
            log(str(num_nans_val))

            log(f"[C{config_id}|F{fold}] --- categorical summary ---")
            cat_nans_train     = X_train[categorical_features].isna().sum()
            cat_unknown_train  = (X_train[categorical_features] == "UNKNOWN").sum()
            log("  NaNs per categorical (train):")
            log(str(cat_nans_train))
            log("  'UNKNOWN' per categorical  (train):")
            log(str(cat_unknown_train))

            log(f"[C{config_id}|F{fold}] distinct values in categorical (train):")
            for col in categorical_features:
                uniques = sorted(X_train[col].dropna().astype(str).unique())
                if len(uniques) > 30:
                    shown = uniques[:30]
                    log(f"    {col}: {len(uniques)} distinct values. First 30: {shown} ...")
                else:
                    log(f"    {col}: {len(uniques)} distinct values: {uniques}")

            unknown_mask_train = (X_train[categorical_features] == "UNKNOWN")
            unknown_mask_val   = (X_val[categorical_features]   == "UNKNOWN")

            unknown_counts_train = unknown_mask_train.sum()
            unknown_counts_val   = unknown_mask_val.sum()

            rows_with_unknown_train = unknown_mask_train.any(axis=1).sum()
            rows_with_unknown_val   = unknown_mask_val.any(axis=1).sum()

            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after (categorical, train):")
            log(str(unknown_counts_train))
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' after  (categorical, val):")
            log(str(unknown_counts_val))
            log(f"[C{config_id}|F{fold}] Num of rows with >=1 UNKNOWN (train): {rows_with_unknown_train}")
            log(f"[C{config_id}|F{fold}] Num of rows with >=1 UNKNOWN (val)  : {rows_with_unknown_val}")

            # ---------------------------------
            # 3) categorical encoding
            # ---------------------------------
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            log(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")

            # 3.1) Target Encoding para Brand e model (FIT WITH LOG TARGET)
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train_log)

            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-Hot Encoding para as restantes categóricas
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])

            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")

            unknown_cols_train = [col for col in X_train_cat.columns if "UNKNOWN" in str(col)]
            unknown_cols_val   = [col for col in X_val_cat.columns if "UNKNOWN" in str(col)]

            # ---------------------------------
            # 4) NUMERICAL SCALING
            # ---------------------------------
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features_fold])
            X_val_num   = scaler.transform(X_val[numeric_features_fold])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features_fold]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features_fold]
            )

            log(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # ---------------------------------
            # 5) concatenate scaled numeric and encoded categorical features
            # ---------------------------------
            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df,   X_val_cat],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # ---------------------------------
            # 6) instantiate and train the random forest regressor for this configuration
            # ---------------------------------
            rf = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **params
            )

            log(f"[C{config_id}|F{fold}] training RandomForestRegressor (log-target)...")
            rf.fit(X_train_final, y_train_log)

            # predictions in log-space -> back-transform to euros for evaluation
            y_pred_train_log = rf.predict(X_train_final)
            y_pred_val_log   = rf.predict(X_val_final)

            y_pred_train = np.expm1(y_pred_train_log)
            y_pred_val   = np.expm1(y_pred_val_log)

            # ---------------------------------
            # 7) compute metrics per fold for both train and validation (EUROS)
            # ---------------------------------
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)

            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)

            log(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.4f} | MAE: {mae_tr:.4f} | R²: {r2_tr:.4f}")
            log(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R²: {r2_val:.4f}")

            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)

            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)

        # average metrics of all folds for this configuration
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)

        mean_rmse_tr = np.mean(fold_rmses_train)
        mean_mae_tr  = np.mean(fold_maes_train)
        mean_r2_tr   = np.mean(fold_r2s_train)

        # combined score giving equal weight to rmse and mae on validation
        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - TRAIN RMSE mean: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f} | R²: {mean_r2_tr:.4f}")
        log(f"Config {config_id} - VAL   RMSE mean: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | R²: {mean_r2_val:.4f}")
        log(f"Config {config_id} - Score combined (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "combo_score": combo_score,
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id} w/ mean RMSE (VAL) = {best_rmse:.4f}")

        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id} w/ mean MAE  (VAL) = {best_mae:.4f}")

        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED] Config {config_id} w/ score = {best_combo:.4f}")

    log("")
    log("")
    log("# =============================")
    log("# END")
    log("# =============================")
    log(f"Best config (min RMSE VAL): {best_config}")
    log(f"best mean RMSE  (VAL): {best_rmse:.4f}")
    log(f"Best config (min MAE VAL): {best_config_mae}")
    log(f"BEST mean MAE (VAL): {best_mae:.4f}")
    log(f"best config (combined score VAL): {best_config_combo}")
    log(f"best combined score (VAL): {best_combo:.4f}")

# -----------------------------
# final summary of the random search (as a dataframe)
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True)

display(results_df_sorted.head(10))

print("\nBest config (min RMSE VAL):")
print(best_config)
print("best mean RMSE (VAL):", best_rmse)

print("\nbest config (min MAE VAL):")
print(best_config_mae)
print("best mean MAE  (VAL):", best_mae)

print("\nbest config found (combined score VAL = 0.5*RMSE + 0.5*MAE):")
print(best_config_combo)
print("best combined score (VAL):", best_combo)

print(f"\nLogs stored at: {log_path}")


# starting random forest
N_SPLITS = 8, N_RANDOM_CONFIGS = 15
param_distributions = {'n_estimators': [200, 400, 600, 800, 1000], 'max_depth': [7, 9, 11, 15, 20], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 4, 6, 8], 'max_features': ['sqrt', 0.33, 0.5, 1.0], 'bootstrap': [True]}
Target transform: log1p(price) during training; metrics computed back in euros via expm1(pred).
Note: previousOwners is not used (excluded from numeric_features and dropped per fold).

 - CONFIG 1/15 
Parameters: {'n_estimators': 400, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 11, 'bootstrap': True}

[CONFIG 1] -- FOLD 1/8
[C1|F1] X_train shape: (66476, 9), X_val shape: (9497, 9)
[C1|F1] y_train shape: (66476,), y_val shape: (9497,)
[C1|F1] NaNs before (just numerical):
year          1306
mileage       1252
engineSize    1344
tax           6940
mpg           6961
dtype: int64
[C1|F1] NaNs before (categorical):
Brand           0
model           0
transmissio

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_train_mean,mae_train_mean,r2_train_mean,rmse_mean,mae_mean,r2_mean,combo_score
6,7,1000,2,2,0.33,20,True,1749.480161,966.646366,0.967712,2374.404603,1306.254733,0.940322,1840.329668
1,2,1000,10,4,0.5,15,True,2061.177150,1162.470137,0.955183,2417.256507,1354.284523,0.938166,1885.770515
8,9,800,10,2,sqrt,20,True,2090.953367,1130.322281,0.953881,2536.632223,1366.789837,0.931911,1951.711030
13,14,600,2,8,0.5,15,True,2294.099830,1260.768494,0.944484,2545.968317,1402.909365,0.931441,1974.438841
11,12,800,10,2,sqrt,15,True,2248.336279,1266.560826,0.946678,2606.150863,1433.723306,0.928153,2019.937085
7,8,1000,2,4,0.5,11,True,2402.812469,1423.442799,0.939099,2617.368366,1518.488523,0.927576,2067.928445
0,1,400,4,2,1.0,11,True,2317.597316,1424.899406,0.943343,2619.098044,1553.028964,0.927483,2086.063504
4,5,200,6,2,1.0,11,True,2341.074701,1429.891774,0.942188,2627.817510,1554.592775,0.926999,2091.205143
5,6,200,10,8,0.5,11,True,2537.427533,1459.905869,0.932084,2707.613463,1541.331755,0.922512,2124.472609
3,4,1000,6,8,sqrt,15,True,2698.245493,1441.976102,0.923204,2882.662057,1540.011788,0.912126,2211.336922



Best config (min RMSE VAL):
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True}
best mean RMSE (VAL): 2374.404603115575

best config (min MAE VAL):
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True}
best mean MAE  (VAL): 1306.254733402416

best config found (combined score VAL = 0.5*RMSE + 0.5*MAE):
{'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.33, 'max_depth': 20, 'bootstrap': True}
best combined score (VAL): 1840.3296682589953

Logs stored at: rf_random_search_log_.txt


# older 

## 3. Final Model Training and Kaggle Submission Generation

In [ ]:
# ----------------------------------------------------
# 0) final configuration chosen manually based on previous random search results (we used the best RMSE one), 
# but all of the obtained configurations are present in 'rf_random_search_log.txt' notebook for further analysis if needed
#    this dictionary stores the hyperparameters that will be used to train the final model on the full training data
# ----------------------------------------------------
""" 
# this is the fs one, and n_estimators was increased to 1000 for fs_1000
final_config = {
    "n_estimators": 200,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.5,
    "max_depth": 20,
    "bootstrap": True,
    "n_features_to_select": 0.9
}
Note: version rf_final_submission_config_fs is the version 11
and rf_final_submission_config_fs_1000est is the 12 
"""

final_config = {
    "n_estimators": 200,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.5,
    "max_depth": 20,
    "bootstrap": True,
    "n_features_to_select": 0.9
}


n_features_pct = final_config.pop("n_features_to_select", 1.0)
print("Config final usada para treino:", final_config)

# ----------------------------------------------------
# 1) create full training copies of features and target
#    we work on copies of X and y to avoid side effects on any previous objects
# ----------------------------------------------------
X_full = X.copy()
y_full = y.copy()


# list of categorical columns for which we want to enforce string normalization
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# apply deterministic string normalization to the selected columns
# this ensures consistent formatting before fitting imputers and resolvers
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])

        X_full = column_string_transformer(
            X_full,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )

invalids = sorted(
    [b for b in X_full['Brand'].unique() if b not in valid_brands],
    key=len
)

# Aplicar a função
X_full, corrections, remaining_invalids = correct_invalid_brands_in_df(
    X_full,
    col='Brand',
    valid_brands=valid_brands,
    invalids=invalids
)

# define high cardinality and low cardinality categorical features as done before
# high cardinality will be target encoded and the remaining categoricals will be one hot encoded (same as before)
high_card_features = ["Brand", "model"]  # features to Target Encoding
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# ----------------------------------------------------
# 2) numerical preprocessing on full training data
#    all fit operations are performed on X_full so that the learned states can be reused on the test set
#    this replicates the same pipeline used inside cross validation but now applied on the entire training data
# ----------------------------------------------------

# 2.1) YEAR
year_state = fit_year_median(
    X_full,
    year_col="year",
    model_col="model"
)
X_full = transform_year_with_model_median(X_full, state=year_state)

# 2.2) MILEAGE
mileage_state = fit_mileage_imputer(
    X_full,
    mileage_col="mileage",
    do_abs=True
)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# 2.3) ENGINE SIZE
engine_state = fit_engine_size_imputer(
    X_full,
    engine_col="engineSize"
)
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# 2.4) TAX
tax_state = fit_tax_imputer(
    X_full,
    tax_col="tax",
    do_abs=True
)
X_full = transform_tax_imputer(X_full, state=tax_state)

# 2.5) MPG
mpg_state = fit_mpg_imputer(
    X_full,
    mpg_col="mpg",
    do_abs=True
)
X_full = transform_mpg_imputer(X_full, state=mpg_state)



# 2.7) PREVIOUS OWNERS
owners_state = fit_previous_owners_imputer(
    X_full,
    owners_col="previousOwners",
    year_col="year",
    mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)


# 2.8) BRANDS
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full,
    valid_brands=valid_brands,
    brand_col="Brand",
    model_col="model",
    year_col="year",
)

X_full, brand_corr_full, brand_still_invalid_full = transform_ambiguous_brands(
    X_full,
    brand_state,
)
print("Após resolver Brand (train full): nº brands ainda inválidas:",
      len(brand_still_invalid_full))

# 2.9) MODELS
model_state = fit_invalid_model_resolver(
    train_df=X_full,
    valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand",
    model_col="model",
    year_col="year",
    fuel_col="fuelType",
    mpg_col="mpg",
)

X_full, model_corr_full, model_still_invalid_full = transform_invalid_models(
    X_full,
    model_state,
)
print("Após resolver model (train full): nº models ainda inválidos:",
      len(model_still_invalid_full))

# 2.10) TRANSMISSION 
transm_state = fit_transmission_resolver(
    train_df=X_full,
    valid_transmissions=valid_transmissions,
    transm_col="transmission",
    brand_col="Brand",
    model_col="model",
    fuel_col="fuelType",
)

X_full, transm_corr_full, transm_still_invalid_full = transform_transmission_resolver(
    X_full,
    transm_state,
)
print("Após resolver transmission (train full): valores distintos:",
      sorted(X_full["transmission"].astype(str).str.upper().unique()))

# 2.11) FUEL TYPE 
fuel_state = fit_fueltype_resolver(
    train_df=X_full,
    valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType",
    brand_col="Brand",
    model_col="model",
    transm_col="transmission",
)

X_full, fuel_corr_full, fuel_still_invalid_full = transform_fueltype_resolver(
    X_full,
    fuel_state,
)
print("Após resolver fuelType (train full): valores distintos:",
      sorted(X_full["fuelType"].astype(str).str.upper().unique()))

# ----------------------------------------------------
# categorical encoding on full training data
#    high cardinality features brand and model are target encoded
#    remaining categorical features are one hot encoded
# ----------------------------------------------------

# fit target encoder on high cardinality features using the full training target
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)

X_full_high = te.transform(X_full[high_card_features]) # transform brand and model into continuous encoded features

# fit one hot encoder on the remaining categorical features
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])

X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1) # combine target encoded and one hot encoded categorical features into a single dataframe

print("X_full_cat shape:", X_full_cat.shape)

# ----------------------------------------------------
# 5) numerical scaling on full training data
#    standardize numerical features so that they have zero mean and unit variance
# ----------------------------------------------------
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

# convert back to dataframe with consistent index and prefixed column names
X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)

print("X_full_num_df shape:", X_full_num_df.shape)


# ----------------------------------------------------
# build final training matrix by concatenating scaled numeric and encoded categorical features
#    will be used to train the final model
# ----------------------------------------------------
X_full_final = pd.concat( [X_full_num_df, X_full_cat], axis=1)

print("X_full_final shape:", X_full_final.shape)


selector = None # Inicializa variavel

if n_features_pct < 1.0 and n_features_pct is not None:
    total_cols = X_full_final.shape[1]
    n_feats_final = int(total_cols * n_features_pct)
    n_feats_final = max(1, n_feats_final)
    
    print(f"A selecionar as melhores {n_feats_final} features...")
    
    # Instancia o seletor (usamos RF rápido para selecionar)
    selector = MyRandomForestSelector(
        n_features=n_feats_final, 
        n_estimators=50, 
        random_state=RANDOM_STATE
    )
    
    # Fit no X_full
    selector.fit(X_full_final, y_full)
    
    # Transform (Reduz colunas)
    X_full_final = selector.transform(X_full_final)
    
    print("Features selecionadas:", selector.selected_features_)
else:
    print("Feature selection ignorado (usando todas as features).")

print("X_full_final shape (DEPOIS seleção):", X_full_final.shape)

# ============================================================
# ----------------------------------------------------
# train final random forest model on full training data using the chosen configuration
#    this is the model that will be used to generate predictions for the test set
# ----------------------------------------------------
rf_final = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config
)

rf_final.fit(X_full_final, y_full)
print("Modelo final treinado com sucesso.")

# ----------------------------------------------------
# prepare test features using the exact same pipeline as for training
#    here we only apply transform operations using the states learned on the training data
# ----------------------------------------------------
test_df = pd.read_csv("../../project_data/test.csv")

# normalize string representation for the same set of categorical columns in the test data
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in test_df.columns:
        # 1. Preencher vazios com UNKNOWN (igual ao treino)
        test_df[col] = fill_unknown(test_df[col])
        
        # 2. Normalizar string
        test_df = column_string_transformer(
            test_df,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )

# select the same base set of numeric and categorical columns as in training
X_test = test_df[numeric_features + categorical_features].copy()

# apply numerical imputers to the test data using training states only
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# resolve categorical inconsistencies on test set using the same states learned from training
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# apply the fitted target encoder and one hot encoder to the test data
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])

X_test_cat = pd.concat([X_test_high, X_test_low], axis=1)

# scale numerical test features using the scaler fitted on the full training data
X_test_num = scaler.transform(X_test[numeric_features])

X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# build final test matrix by concatenating scaled numeric and encoded categorical test features
X_test_final = pd.concat(
    [X_test_num_df, X_test_cat],
    axis=1
)

if selector is not None:
    # Usa o selector que já foi treinado (fitted) no X_full
    X_test_final = selector.transform(X_test_final)
    print("Seleção aplicada ao teste.")

print("X_test_final shape (DEPOIS seleção):", X_test_final.shape)
# ----------------------------------------------------
# generate test set predictions and build submission file
#    predictions are first inspected as floats and then optionally rounded to integers
# ----------------------------------------------------
y_test_pred = rf_final.predict(X_test_final)

print("Primeiras 10 previsões (float):", y_test_pred[:10])
print("Resumo das previsões (float):")
print(pd.Series(y_test_pred).describe())

# prices round the predictions to the nearest integer
y_test_pred_round = np.round(y_test_pred).astype(int)

# carid and predicted price
submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred
})

submission_path = "rf_final_submission_config_fs.csv"
submission.to_csv(submission_path, index=False)

print("Ficheiro de submissão criado em:", submission_path)


Config final usada para treino: {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Após resolver Brand (train full): nº brands ainda inválidas: 0
Após resolver model (train full): nº models ainda inválidos: 1
Após resolver transmission (train full): valores distintos: ['AUTOMATIC', 'MANUAL', 'SEMIAUTO']
Após resolver fuelType (train full): valores distintos: ['DIESEL', 'ELECTRIC', 'HYBRID', 'OTHER', 'PETROL']
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 7)
X_full_final shape: (75973, 17)
A selecionar as melhores 15 features...
Features selecionadas: ['model', 'num_mileage', 'num_engineSize', 'num_year', 'num_mpg', 'num_paintQuality%', 'Brand', 'num_tax', 'transmission_MANUAL', 'num_previousOwners', 'fuelType_PETROL', 'fuelType_DIESEL', 'transmission_SEMIAUTO', 'transmission_AUTOMATIC', 'fuelType_HYBRID']
X_full_final shape (DEPOIS seleção): (75973, 15)
Modelo final treinado com sucesso.
Seleção a

## WITH GRID SERACH AND FEATURE SELECTION

In [ ]:
# ---------------------------------------------------------
# CORE CONFIGURATION
# ---------------------------------------------------------
N_SPLITS = 8 
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS, 
    shuffle=True,  
    random_state=RANDOM_STATE 
)

# ---------------------------------------------------------
# HYPERPARAMETERS
# ---------------------------------------------------------
param_distributions = {
    "n_estimators": [200, 400, 600, 800, 1000], 
    "max_depth": [7, 9, 11, 15, 20],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "max_features": ["sqrt", 0.33, 0.5, 1.0], 
    "bootstrap": [True], 
    "n_features_to_select": [1.0, 0.9, 0.8, 0.6, 0.4] 
}

# ---------------------------------------------------------
# RANDOM SEARCH CONFIG
# ---------------------------------------------------------
N_RANDOM_CONFIGS = 250 

sampler = ParameterSampler(
    param_distributions, 
    n_iter=N_RANDOM_CONFIGS, 
    random_state=RANDOM_STATE 
)

search_results = [] 

best_rmse = np.inf 
best_config = None 

best_mae = np.inf
best_config_mae = None

best_combo = np.inf
best_config_combo = None

log_path = "rf_250_random_search_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    log("# =============================")
    log(f"# START OF RF RANDOM SEARCH (DETAILED LOGS)")
    log(f"# {N_RANDOM_CONFIGS} ITERATIONS")
    log("# =============================")
    
    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        # Separate params
        frac_feats = params.get("n_features_to_select", 1.0)
        rf_params = {k: v for k, v in params.items() if k != "n_features_to_select"}

        # Metrics Lists
        fold_rmses, fold_maes, fold_r2s = [], [], []
        fold_rmses_train, fold_maes_train, fold_r2s_train = [], [], []
        fold_med_ae, fold_mape, fold_bias = [], [], []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # --- 1) Split Data ---
            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log(f"\n[C{config_id}|F{fold}] Processing fold...")
            
            # --- LOG: Initial State ---
            log(f"  > Initial NaNs (Train Numeric): {X_train[numeric_features].isna().sum().sum()}")
            log(f"  > Initial NaNs (Train Categorical): {X_train[categorical_features].isna().sum().sum()}")
            unknowns_init = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"  > Initial 'UNKNOWN' counts:\n{unknowns_init[unknowns_init > 0]}")

            # --- 2) Numerical Preprocessing ---
            # 2.1 Year
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2 Mileage
            mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3 Engine Size
            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # 2.4 Tax
            tax_state = fit_tax_imputer(X_train, tax_col="tax", do_abs=True)
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            # 2.5 MPG
            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            
            
            # 2.7 Previous Owners
            owners_state = fit_previous_owners_imputer(
                X_train, owners_col="previousOwners", year_col="year", mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            # --- Categorical Resolvers ---
            # 2.8 Brand
            brand_state = fit_ambiguous_brand_resolver(
                X_train, valid_brands=valid_brands, brand_col="Brand", model_col="model", year_col="year"
            )
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            # 2.9 Model
            model_state = fit_invalid_model_resolver(
                X_train, valid_models_by_brand=valid_models_by_brand, 
                brand_col="Brand", model_col="model", year_col="year", fuel_col="fuelType", mpg_col="mpg"
            )
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            # 2.10 Transmission
            transm_state = fit_transmission_resolver(
                X_train, valid_transmissions=valid_transmissions, transm_col="transmission"
            )
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            # 2.11 Fuel Type
            fuel_state = fit_fueltype_resolver(
                X_train, valid_fueltypes=valid_fueltypes, fuel_col="fuelType"
            )
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            # --- LOG: Post-Processing State ---
            log(f"  > Post-Proc NaNs (Train Numeric): {X_train[numeric_features].isna().sum().sum()}")
            unknowns_post = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"  > Post-Proc 'UNKNOWN' counts:\n{unknowns_post[unknowns_post > 0]}")

            # --- 3) Categorical Encoding ---
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # --- 4) Numerical Scaling ---
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features])
            X_val_num   = scaler.transform(X_val[numeric_features])

            X_train_num_df = pd.DataFrame(X_train_num, index=X_train.index, columns=[f"num_{col}" for col in numeric_features])
            X_val_num_df = pd.DataFrame(X_val_num, index=X_val.index, columns=[f"num_{col}" for col in numeric_features])

            # --- 5) Concatenate Features ---
            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df, X_val_cat],   axis=1)
            
            # ============================================================
            # FEATURE SELECTION (Percentage Logic)
            # ============================================================
            total_cols = X_train_final.shape[1]
            if frac_feats >= 1.0:
                n_feats_final = total_cols
            else:
                n_feats_final = int(total_cols * frac_feats)
                n_feats_final = max(1, n_feats_final)

            if n_feats_final < total_cols:
                log(f"  > Feature Selection: Keeping top {n_feats_final} columns ({frac_feats*100:.0f}%). Dropping {total_cols - n_feats_final} columns.")
                selector = MyRandomForestSelector(n_features=n_feats_final, n_estimators=50, random_state=RANDOM_STATE)
                selector.fit(X_train_final, y_train)
                
                dropped_cols = [c for c in X_train_final.columns if c not in selector.selected_features_]
                # Log dropped columns (limited to first 10 to avoid spam)
                log(f"  > Dropped Columns (First 10): {dropped_cols[:10]} ...")
                
                X_train_final = selector.transform(X_train_final)
                X_val_final   = selector.transform(X_val_final)
            else:
                log(f"  > Feature Selection: Skipped (Using all {total_cols} features).")
            # ============================================================

            # --- 6) Train Model ---
            rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **rf_params)
            rf.fit(X_train_final, y_train)

            y_pred_train = rf.predict(X_train_final)
            y_pred_val   = rf.predict(X_val_final)

            # --- 7) METRICS ---
            mse_val, mae_val = mean_squared_error(y_val, y_pred_val), mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            r2_val = r2_score(y_val, y_pred_val)

            mse_tr, mae_tr = mean_squared_error(y_train, y_pred_train), mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            r2_tr = r2_score(y_train, y_pred_train)
            
            med_ae_val = median_absolute_error(y_val, y_pred_val)
            mape_val = mean_absolute_percentage_error(y_val, y_pred_val) * 100 
            bias_val = np.mean(y_val - y_pred_val)

            log(f"  > Fold Results: [TRAIN] RMSE={rmse_tr:.2f}, MAE={mae_tr:.2f} | [VAL] RMSE={rmse_val:.2f}, MAE={mae_val:.2f}")

            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)
            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)
            fold_med_ae.append(med_ae_val)
            fold_mape.append(mape_val)
            fold_bias.append(bias_val)

        # Average metrics
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)
        mean_rmse_tr  = np.mean(fold_rmses_train)
        mean_mae_tr   = np.mean(fold_maes_train)
        mean_r2_tr    = np.mean(fold_r2s_train)
        mean_med_ae_val = np.mean(fold_med_ae)
        mean_mape_val   = np.mean(fold_mape)
        mean_bias_val   = np.mean(fold_bias)

        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - [VAL AVG] RMSE: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | MedAE: {mean_med_ae_val:.4f} | Bias: {mean_bias_val:.4f}")
        log(f"Config {config_id} - [TRAIN AVG] RMSE: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "med_ae_mean": mean_med_ae_val,
            "mape_mean": mean_mape_val,
            "bias_mean": mean_bias_val,
            "combo_score": combo_score,
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id}")
            
        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED SCORE] Config {config_id}")

    log("# END OF RANDOM SEARCH RF")

# -----------------------------
# Results
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True)

cols_to_display = ["config_id", "n_estimators", "max_depth", "n_features_to_select", "rmse_mean", "mae_mean", "med_ae_mean", "mape_mean", "bias_mean"]
cols_to_display = [c for c in cols_to_display if c in results_df_sorted.columns]

print("\nTop 10 Configurations by RMSE:")
display(results_df_sorted[cols_to_display].head(10))

print("\n" + "="*50)
print("FINAL RESULTS ANALYSIS")
print("="*50)

# 1. Best RMSE
print("\n--- BEST CONFIGURATION BY RMSE (Root Mean Squared Error) ---")
best_row_rmse = results_df_sorted.iloc[0]
print(f"Config ID: {best_row_rmse['config_id']}")
print(f"Params: {best_config}")
print(f"RMSE: {best_row_rmse['rmse_mean']:.4f}")

# 2. Best MAE
print("\n--- BEST CONFIGURATION BY MAE (Mean Absolute Error) ---")
best_row_mae = results_df.sort_values(by="mae_mean", ascending=True).iloc[0]
print(f"Config ID: {best_row_mae['config_id']}")
print(f"Params: {best_config_mae}")
print(f"MAE: {best_row_mae['mae_mean']:.4f}")

# 3. Best Median Absolute Error
print("\n--- BEST CONFIGURATION BY MEDIAN AE (Robustness Check) ---")
best_row_med = results_df.sort_values(by="med_ae_mean", ascending=True).iloc[0]
best_params_med = {k: best_row_med[k] for k in param_distributions.keys() if k in best_row_med}
print(f"Config ID: {best_row_med['config_id']}")
print(f"Params: {best_params_med}")
print(f"Median AE: {best_row_med['med_ae_mean']:.4f}")

print(f"\nDetailed logs saved at: {log_path}")

# =============================
# START OF RF RANDOM SEARCH (DETAILED LOGS)
# 250 ITERATIONS
# =============================

######## CONFIG 1/250 ########
Parameters: {'n_features_to_select': 0.8, 'n_estimators': 200, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 15, 'bootstrap': True}

[C1|F1] Processing fold...
  > Initial NaNs (Train Numeric): 20456
  > Initial NaNs (Train Categorical): 0
  > Initial 'UNKNOWN' counts:
Brand           1328
model           1314
transmission    1944
fuelType        1296
dtype: Int64
  > Post-Proc NaNs (Train Numeric): 0
  > Post-Proc 'UNKNOWN' counts:
Series([], dtype: int64)
  > Feature Selection: Keeping top 13 columns (80%). Dropping 4 columns.
  > Dropped Columns (First 10): ['transmission_AUTOMATIC', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER'] ...
  > Fold Results: [TRAIN] RMSE=2113.68, MAE=1247.43 | [VAL] RMSE=2265.38, MAE=1433.38

[C1|F2] Processing fold...
  > Initial NaNs (Train Numeric): 2

,config_id,n_estimators,max_depth,n_features_to_select,rmse_mean,mae_mean,med_ae_mean,mape_mean,bias_mean
88,89,200,20,0.9,2186.695353,1283.851822,808.170233,8.119555,-9.208409
18,19,200,15,0.9,2252.288174,1332.312032,852.154184,8.425247,-1.985789
108,109,1000,20,0.6,2253.914894,1306.799506,816.594399,8.218594,-2.841305
168,169,1000,20,1.0,2262.911298,1317.007693,832.903596,8.459663,-1.084473
142,143,800,15,0.8,2265.962852,1347.482110,868.638947,8.622752,0.531803
162,163,400,15,0.8,2273.985072,1346.756403,846.905252,8.430676,-5.020219
172,173,400,15,0.6,2277.242314,1343.864809,853.439974,8.455649,-3.938415
226,227,200,20,1.0,2278.304672,1323.524003,836.435158,8.483004,-1.758605
57,58,400,20,1.0,2287.602066,1317.764486,817.678036,8.241558,-4.373663
78,79,200,20,1.0,2288.566513,1324.138134,838.042976,8.462163,-2.491834



FINAL RESULTS ANALYSIS

--- BEST CONFIGURATION BY RMSE (Root Mean Squared Error) ---
Config ID: 89
Params: {'n_features_to_select': 0.9, 'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True}
RMSE: 2186.6954

--- BEST CONFIGURATION BY MAE (Mean Absolute Error) ---
Config ID: 89
Params: {'n_features_to_select': 0.9, 'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True}
MAE: 1283.8518

--- BEST CONFIGURATION BY MEDIAN AE (Robustness Check) ---
Config ID: 89
Params: {'n_estimators': np.int64(200), 'max_depth': np.int64(20), 'min_samples_split': np.int64(2), 'min_samples_leaf': np.int64(1), 'max_features': 0.5, 'bootstrap': np.True_, 'n_features_to_select': np.float64(0.9)}
Median AE: 808.1702

Detailed logs saved at: rf_250_random_search_log.txt


## TEST 

In [ ]:
# ----------------------------------------------------
# 0) final configuration chosen manually based on previous random search results (we used the best RMSE one), 
# but all of the obtained configurations are present in 'rf_random_search_log.txt' notebook for further analysis if needed
#    this dictionary stores the hyperparameters that will be used to train the final model on the full training data
# ----------------------------------------------------
final_config = {
    "n_estimators": 1000,
    "min_samples_split": 6,
    "min_samples_leaf": 1,
    "max_features": None,
    "max_depth": 15,
    "bootstrap": True,
}

n_features_pct = final_config.pop("n_features_to_select", 1.0) # Remove do dict para não dar erro no RF
print(f"Config final RF: {final_config}")
print(f"Features a manter: {n_features_pct * 100}%")

# ----------------------------------------------------
# 1) create full training copies of features and target
#    we work on copies of X and y to avoid side effects on any previous objects
# ----------------------------------------------------
X_full = X.copy()
y_full = y.copy()


# list of categorical columns for which we want to enforce string normalization
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# apply deterministic string normalization to the selected columns
# this ensures consistent formatting before fitting imputers and resolvers
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])

        X_full = column_string_transformer(
            X_full,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )

invalids = sorted(
    [b for b in X_full['Brand'].unique() if b not in valid_brands],
    key=len
)

# Aplicar a função
X_full, corrections, remaining_invalids = correct_invalid_brands_in_df(
    X_full,
    col='Brand',
    valid_brands=valid_brands,
    invalids=invalids
)

print(f"Correções aplicadas: {len(corrections)}")
print(f"Brands ainda inválidas (serão tratadas pelo resolver robusto): {len(remaining_invalids)}")


# define high cardinality and low cardinality categorical features as done before
# high cardinality will be target encoded and the remaining categoricals will be one hot encoded (same as before)
high_card_features = ["Brand", "model"]  # features to Target Encoding
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# ----------------------------------------------------
# 2) numerical preprocessing on full training data
#    all fit operations are performed on X_full so that the learned states can be reused on the test set
#    this replicates the same pipeline used inside cross validation but now applied on the entire training data
# ----------------------------------------------------

# 2.1) YEAR
year_state = fit_year_median(
    X_full,
    year_col="year",
    model_col="model"
)
X_full = transform_year_with_model_median(X_full, state=year_state)

# 2.2) MILEAGE
mileage_state = fit_mileage_imputer(
    X_full,
    mileage_col="mileage",
    do_abs=True
)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# 2.3) ENGINE SIZE
engine_state = fit_engine_size_imputer(
    X_full,
    engine_col="engineSize"
)
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# 2.4) TAX
tax_state = fit_tax_imputer(
    X_full,
    tax_col="tax",
    do_abs=True
)
X_full = transform_tax_imputer(X_full, state=tax_state)

# 2.5) MPG
mpg_state = fit_mpg_imputer(
    X_full,
    mpg_col="mpg",
    do_abs=True
)
X_full = transform_mpg_imputer(X_full, state=mpg_state)




# 2.7) PREVIOUS OWNERS
owners_state = fit_previous_owners_imputer(
    X_full,
    owners_col="previousOwners",
    year_col="year",
    mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)


# 2.8) BRANDS
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full,
    valid_brands=valid_brands,
    brand_col="Brand",
    model_col="model",
    year_col="year",
)

X_full, brand_corr_full, brand_still_invalid_full = transform_ambiguous_brands(
    X_full,
    brand_state,
)
print("Após resolver Brand (train full): nº brands ainda inválidas:",
      len(brand_still_invalid_full))

# 2.9) MODELS
model_state = fit_invalid_model_resolver(
    train_df=X_full,
    valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand",
    model_col="model",
    year_col="year",
    fuel_col="fuelType",
    mpg_col="mpg",
)

X_full, model_corr_full, model_still_invalid_full = transform_invalid_models(
    X_full,
    model_state,
)
print("Após resolver model (train full): nº models ainda inválidos:",
      len(model_still_invalid_full))

# 2.10) TRANSMISSION 
transm_state = fit_transmission_resolver(
    train_df=X_full,
    valid_transmissions=valid_transmissions,
    transm_col="transmission",
    brand_col="Brand",
    model_col="model",
    fuel_col="fuelType",
)

X_full, transm_corr_full, transm_still_invalid_full = transform_transmission_resolver(
    X_full,
    transm_state,
)
print("Após resolver transmission (train full): valores distintos:",
      sorted(X_full["transmission"].astype(str).str.upper().unique()))

# 2.11) FUEL TYPE 
fuel_state = fit_fueltype_resolver(
    train_df=X_full,
    valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType",
    brand_col="Brand",
    model_col="model",
    transm_col="transmission",
)

X_full, fuel_corr_full, fuel_still_invalid_full = transform_fueltype_resolver(
    X_full,
    fuel_state,
)
print("Após resolver fuelType (train full): valores distintos:",
      sorted(X_full["fuelType"].astype(str).str.upper().unique()))

# ----------------------------------------------------
# categorical encoding on full training data
#    high cardinality features brand and model are target encoded
#    remaining categorical features are one hot encoded
# ----------------------------------------------------

# fit target encoder on high cardinality features using the full training target
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)

X_full_high = te.transform(X_full[high_card_features]) # transform brand and model into continuous encoded features

# fit one hot encoder on the remaining categorical features
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])

X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1) # combine target encoded and one hot encoded categorical features into a single dataframe

print("X_full_cat shape:", X_full_cat.shape)

# ----------------------------------------------------
# 5) numerical scaling on full training data
#    standardize numerical features so that they have zero mean and unit variance
# ----------------------------------------------------
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

# convert back to dataframe with consistent index and prefixed column names
X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)

print("X_full_num_df shape:", X_full_num_df.shape)


# ----------------------------------------------------
# build final training matrix by concatenating scaled numeric and encoded categorical features
#    will be used to train the final model
# ----------------------------------------------------
X_full_final = pd.concat(
    [X_full_num_df, X_full_cat],
    axis=1
)

print("X_full_final shape:", X_full_final.shape)


# ----------------------------------------------------
# train final random forest model on full training data using the chosen configuration
#    this is the model that will be used to generate predictions for the test set
# ----------------------------------------------------
rf_final = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config
)

rf_final.fit(X_full_final, y_full)
print("Modelo final treinado com sucesso.")

# ----------------------------------------------------
# prepare test features using the exact same pipeline as for training
#    here we only apply transform operations using the states learned on the training data
# ----------------------------------------------------
test_df = pd.read_csv("../../project_data/test.csv")

# normalize string representation for the same set of categorical columns in the test data
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in test_df.columns:
        test_df = column_string_transformer(
            test_df,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )


# select the same base set of numeric and categorical columns as in training
X_test = test_df[numeric_features + categorical_features].copy()

# apply numerical imputers to the test data using training states only
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# resolve categorical inconsistencies on test set using the same states learned from training
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# apply the fitted target encoder and one hot encoder to the test data
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])

X_test_cat = pd.concat([X_test_high, X_test_low], axis=1)

# scale numerical test features using the scaler fitted on the full training data
X_test_num = scaler.transform(X_test[numeric_features])

X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# build final test matrix by concatenating scaled numeric and encoded categorical test features
X_test_final = pd.concat(
    [X_test_num_df, X_test_cat],
    axis=1
)

print("X_test_final shape:", X_test_final.shape)

# ----------------------------------------------------
# generate test set predictions and build submission file
#    predictions are first inspected as floats and then optionally rounded to integers
# ----------------------------------------------------
y_test_pred = rf_final.predict(X_test_final)

print("Primeiras 10 previsões (float):", y_test_pred[:10])
print("Resumo das previsões (float):")
print(pd.Series(y_test_pred).describe())

# prices round the predictions to the nearest integer
y_test_pred_round = np.round(y_test_pred).astype(int)

# carid and predicted price
submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred
})

submission_path = "rf_final_submission_config_1000_6_1_15_d.csv"
submission.to_csv(submission_path, index=False)

print("Ficheiro de submissão criado em:", submission_path)


Config final RF: {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Features a manter: 100.0%
Correções aplicadas: 0
Brands ainda inválidas (serão tratadas pelo resolver robusto): 2
Após resolver Brand (train full): nº brands ainda inválidas: 0
Após resolver model (train full): nº models ainda inválidos: 1
Após resolver transmission (train full): valores distintos: ['AUTOMATIC', 'MANUAL', 'SEMIAUTO']
Após resolver fuelType (train full): valores distintos: ['DIESEL', 'ELECTRIC', 'HYBRID', 'OTHER', 'PETROL']
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 7)
X_full_final shape: (75973, 17)
Modelo final treinado com sucesso.
X_test_final shape: (32567, 17)
Primeiras 10 previsões (float): [ 8385.93548445 21923.69317011 14335.61472473 16777.98394774
 23704.29658898 10575.89043131 13355.48938839 14781.77002539
  4870.99690598 17846.67278988]
Resumo das previsões (float):
count     32567.000000
mean      1

In [4]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# at this point, this are all the features in use 
categorical_features = ['Brand', 'model', 'transmission', 'fuelType']              
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

N_SPLITS = 8 # the number of folds for K-Fold cross-validation
RANDOM_STATE = 42 # seed to control randomness

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

# ==============================================================================
# 0) SETTINGS
# ==============================================================================
RANDOM_STATE = 42

# hiperparâmetros 
final_config = {
    "n_estimators": 1000,
    "min_samples_split": 6,
    "min_samples_leaf": 1,
    "max_features": None,
    "max_depth": 15,
    "bootstrap": True,
}


FS_KEEP_RATIO = 0.65
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

print(f"- Config final RF: {final_config}")
print(f"- FS_KEEP_RATIO: {FS_KEEP_RATIO}")

# ==============================================================================
# 1) LOAD TEST + IDS (igual ao HGB)
# ==============================================================================
try:
    test_df_raw = pd.read_csv("test.csv")
except:
    test_df_raw = pd.read_csv("../../project_data/test.csv")

ID_CANDIDATES = ["id", "carID", "carId", "car_id"]
ID_COL = next((c for c in ID_CANDIDATES if c in test_df_raw.columns), None)

if ID_COL is not None:
    test_ids = test_df_raw[ID_COL].copy()
    test_df = test_df_raw.drop(columns=[ID_COL]).copy()
else:
    test_ids = test_df_raw.index.copy()
    test_df = test_df_raw.copy()

# ==============================================================================
# 2) START DATA 
# ==============================================================================
X_full = X.copy()
y_full = y.copy()
y_full_log = np.log1p(y_full)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]


# ==============================================================================
# 4) STRING NORMALIZATION (TRAIN + TEST) 
# ==============================================================================
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer(
            X_full, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(
            test_df, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

# Restrict to expected columns (train schema)
expected_cols = [c for c in (numeric_features + categorical_features) if c in X_full.columns]
X_full = X_full[expected_cols].copy()
X_test = test_df.reindex(columns=expected_cols, fill_value=np.nan).copy()

# ==============================================================================
# 5) FIT + TRANSFORM ON FULL TRAIN (igual ao HGB)
# ==============================================================================
print("- fitting transforms on full train")

year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# IMPORTANT: owners imputer must happen BEFORE dropping previousOwners
owners_state = fit_previous_owners_imputer(X_full, "previousOwners", "year", "mileage")
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 6) FEATURE ENGINEERING (FULL TRAIN) 
# ==============================================================================
print("- creating engineered features on full train")

X_full = add_owners_flagged(X_full, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_full = create_age_and_drop_year(X_full, year_col="year", base_year=2020)
X_full = add_mileage_features(X_full, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_full = add_engine_bins(X_full, engine_col="engineSize", new_col="engine_bin")

# engine_bin as low-card categorical
X_full["engine_bin"] = X_full["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
low_card_curr = low_card_features + ["engine_bin"]

# ==============================================================================
# 7) ENCODING (FIT ON FULL TRAIN)
# ==============================================================================
print("- fitting encoders on full train")

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full_log)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_curr])
X_full_low = ohe.transform(X_full[low_card_curr])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr = [c for c in X_full.columns if c not in drop_for_numeric]

X_full_final = pd.concat([X_full[numeric_features_curr], X_full_cat], axis=1)
print(f"- train matrix shape: {X_full_final.shape}")

# ==============================================================================
# 8) FEATURE SELECTION (FIT ON FULL TRAIN) 
# ==============================================================================
print("- fitting feature selector (RF)")

n_feats = X_full_final.shape[1]
k = int(np.ceil(FS_KEEP_RATIO * n_feats))
k = max(1, min(k, n_feats))

rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
rf_fs.fit(X_full_final, y_full_log)

selector = SelectFromModel(
    estimator=rf_fs,
    threshold=-np.inf,
    max_features=k,
    prefit=True
)

selected_cols = X_full_final.columns[selector.get_support()]
X_full_sel = X_full_final[selected_cols]
print(f"- selected features: {len(selected_cols)}/{n_feats}")

# ==============================================================================
# 9) TRAIN FINAL RF (LOG TARGET)  
# ==============================================================================
print("- training final RF (log target)")

rf_final = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config
)
rf_final.fit(X_full_sel, y_full_log)
print("- model trained")

# ==============================================================================
# 10) TRANSFORM TEST 
# ==============================================================================
print("- transforming test set")

X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_custom_rules(X_test, "tax", "year", "fuelType", "engineSize")
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test = transform_previous_owners_imputer(X_test, state=owners_state)

X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

X_test = add_owners_flagged(X_test, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_test = create_age_and_drop_year(X_test, year_col="year", base_year=2020)
X_test = add_mileage_features(X_test, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_test = add_engine_bins(X_test, engine_col="engineSize", new_col="engine_bin")

X_test["engine_bin"] = X_test["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

# encoding transform-only
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_curr])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr_test = [c for c in X_test.columns if c not in drop_for_numeric]

X_test_final = pd.concat([X_test[numeric_features_curr_test], X_test_cat], axis=1)

# align to train columns before selecting
X_test_final = X_test_final.reindex(columns=X_full_final.columns, fill_value=0)

# apply same selected columns
X_test_sel = X_test_final.reindex(columns=selected_cols, fill_value=0)

print(f"- test matrix shape: {X_test_sel.shape}")

# ==============================================================================
# 11) PREDICT + SUBMISSION
# ==============================================================================
pred_log = rf_final.predict(X_test_sel)
pred_final = np.expm1(pred_log)
pred_final = np.maximum(pred_final, 0)

submission = pd.DataFrame({
    (ID_COL if ID_COL is not None else "id"): test_ids,
    "price": pred_final
})

submission_filename = "submission_rf_log_target_fs065.csv"
submission.to_csv(submission_filename, index=False)

print(f"- saved: {submission_filename}")
print(submission.head())


- Config final RF: {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
- FS_KEEP_RATIO: 0.65
- fitting transforms on full train
- creating engineered features on full train
- fitting encoders on full train
- train matrix shape: (75973, 25)
- fitting feature selector (RF)
- selected features: 17/25
- training final RF (log target)
- model trained
- transforming test set
- test matrix shape: (32567, 17)
- saved: submission_rf_log_target_fs065.csv
    carID         price
0   89856  11583.228126
1  106581  24982.627453
2   80886  14166.922095
3  100174  16619.799909
4   81376  23085.463528


# Ver

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

numeric_features = ["year", "mileage", "engineSize", "tax", "mpg"]

# -----------------------------
# Helpers (métricas em €)
# -----------------------------
def _rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def _bias(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

# -----------------------------
# CONFIG (a tua)
# -----------------------------
RANDOM_STATE = 42

final_config = {
    "n_estimators": 1000,
    "min_samples_split": 6,
    "min_samples_leaf": 1,
    "max_features": None,
    "max_depth": 15,
    "bootstrap": True,
}

N_SPLITS = 8
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]
low_card_curr = low_card_features  # sem engine_bin

DROP_FROM_MODEL = ["previousOwners"]
numeric_features_model = ["mileage", "engineSize", "tax", "mpg", "age"]  # year -> age

print(f"\n--- 8-FOLD CV | RF (log target) | AGE only | no previousOwners | params={final_config} ---\n")

rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    # -----------------------------
    # Split
    # -----------------------------
    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val   = y.iloc[val_idx].copy()

    # log-target só no treino
    y_train_log = np.log1p(y_train.astype(float))

    # -----------------------------
    # String normalization (fold-wise)
    # -----------------------------
    for col in cols_to_normalize:
        if col in X_train.columns:
            X_train[col] = fill_unknown(X_train[col])
            X_train = column_string_transformer(
                X_train, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
        if col in X_val.columns:
            X_val[col] = fill_unknown(X_val[col])
            X_val = column_string_transformer(
                X_val, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )

    # Restrict to expected schema
    expected_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
    X_train = X_train[expected_cols].copy()
    X_val   = X_val.reindex(columns=expected_cols, fill_value=np.nan).copy()

    # -----------------------------
    # Fit + transform (NO LEAKAGE)
    # -----------------------------
    year_state = fit_year_median(X_train, year_col="year", model_col="model")
    X_train = transform_year_with_model_median(X_train, state=year_state)
    X_val   = transform_year_with_model_median(X_val,   state=year_state)

    mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
    X_train = transform_mileage_imputer(X_train, state=mileage_state)
    X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

    engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
    X_train = transform_engine_size_imputer(X_train, state=engine_state)
    X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

    X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
    X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

    mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
    X_train = transform_mpg_imputer(X_train, state=mpg_state)
    X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

    brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
    X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
    X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

    model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
    X_train, _, _ = transform_invalid_models(X_train, model_state)
    X_val,   _, _ = transform_invalid_models(X_val,   model_state)

    transm_state = fit_transmission_resolver(X_train, valid_transmissions)
    X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
    X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

    fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
    X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
    X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

    # -----------------------------
    # Feature engineering: ONLY AGE (drop year)
    # -----------------------------
    X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
    X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

    # hard guarantee: previousOwners is not used
    X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
    X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

    # -----------------------------
    # Encoding (fit no train com LOG target)
    # -----------------------------
    te = MyTargetEncoder(smoothing=5)
    te.fit(X_train[high_card_features], y_train_log)
    X_train_high = te.transform(X_train[high_card_features])
    X_val_high   = te.transform(X_val[high_card_features])

    if len(low_card_curr) > 0:
        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])
    else:
        X_train_low = pd.DataFrame(index=X_train.index)
        X_val_low   = pd.DataFrame(index=X_val.index)

    X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
    X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

    drop_for_numeric = set(high_card_features + low_card_curr)
    numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]
    numeric_features_curr = [c for c in numeric_features_model if c in numeric_features_curr]

    X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
    X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

    X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

    # ---- imprime features usadas neste fold
    feat_names = list(X_train_final.columns)
    print(f"\n[Fold {fold}] Features used ({len(feat_names)}):")
    print(feat_names)

    # -----------------------------
    # Train RF (log target) + predict (back to €)
    # -----------------------------
    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **final_config
    )
    rf.fit(X_train_final, y_train_log)

    pred_tr_log  = rf.predict(X_train_final)
    pred_val_log = rf.predict(X_val_final)

    pred_tr  = np.expm1(pred_tr_log)
    pred_val = np.expm1(pred_val_log)

    # -----------------------------
    # Metrics (em €)
    # -----------------------------
    tr_rmse = _rmse(y_train, pred_tr)
    tr_mae  = float(mean_absolute_error(y_train, pred_tr))
    tr_r2   = float(r2_score(y_train, pred_tr))
    tr_bias = _bias(y_train, pred_tr)

    val_rmse = _rmse(y_val, pred_val)
    val_mae  = float(mean_absolute_error(y_val, pred_val))
    val_r2   = float(r2_score(y_val, pred_val))
    val_bias = _bias(y_val, pred_val)

    print(
        f"[Fold {fold}] "
        f"TRAIN -> RMSE: {tr_rmse:.1f} | MAE: {tr_mae:.1f} | R2: {tr_r2:.4f} | Bias: {tr_bias:.1f}   ||   "
        f"VAL -> RMSE: {val_rmse:.1f} | MAE: {val_mae:.1f} | R2: {val_r2:.4f} | Bias: {val_bias:.1f}"
    )

    rows.append({
        "fold": fold,
        "n_features": int(X_train_final.shape[1]),
        "train_rmse": tr_rmse, "train_mae": tr_mae, "train_r2": tr_r2, "train_bias": tr_bias,
        "val_rmse": val_rmse, "val_mae": val_mae, "val_r2": val_r2, "val_bias": val_bias,
    })

# -----------------------------
# Summary
# -----------------------------
results_df = pd.DataFrame(rows)

print("\n--- CV SUMMARY (mean ± std) ---")
for split in ["train", "val"]:
    print(
        f"{split.upper()} | "
        f"RMSE: {results_df[f'{split}_rmse'].mean():.1f} ± {results_df[f'{split}_rmse'].std(ddof=1):.1f} | "
        f"MAE: {results_df[f'{split}_mae'].mean():.1f} ± {results_df[f'{split}_mae'].std(ddof=1):.1f} | "
        f"R2: {results_df[f'{split}_r2'].mean():.4f} ± {results_df[f'{split}_r2'].std(ddof=1):.4f} | "
        f"Bias: {results_df[f'{split}_bias'].mean():.1f} ± {results_df[f'{split}_bias'].std(ddof=1):.1f}"
    )

display(results_df)



--- 8-FOLD CV | RF (log target) | AGE only | no previousOwners | params={'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True} ---


[Fold 1] Features used (15):
['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
[Fold 1] TRAIN -> RMSE: 1745.0 | MAE: 1039.2 | R2: 0.9681 | Bias: 114.6   ||   VAL -> RMSE: 2103.9 | MAE: 1332.1 | R2: 0.9511 | Bias: 91.1

[Fold 2] Features used (15):
['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
[Fold 2] TRAIN -> RMSE: 1746.7 | MAE: 1039.7 | R2: 0.9679 | Bias: 113.2   ||   VAL -> RMSE: 2343.9 | MAE: 1367.3 | R2: 0.9414

,fold,n_features,train_rmse,train_mae,train_r2,train_bias,val_rmse,val_mae,val_r2,val_bias
0,1,15,1744.960448,1039.173426,0.968090,114.596019,2103.878465,1332.103773,0.951091,91.079585
1,2,15,1746.704109,1039.728083,0.967866,113.208160,2343.862650,1367.308448,0.941447,119.423555
2,3,15,1734.199170,1032.873679,0.968320,113.221556,2410.718415,1355.351559,0.938125,178.916328
3,4,15,1655.063261,1033.112144,0.970757,110.980652,2838.539131,1386.282361,0.921565,179.138445
4,5,15,1740.863506,1037.571909,0.967966,113.826755,2283.671348,1347.484257,0.945790,127.511163
5,6,15,1739.568971,1037.530622,0.968005,112.710925,2338.557589,1336.639970,0.943262,170.853108
6,7,15,1731.376786,1033.695028,0.968390,112.756367,2205.322269,1324.603509,0.948594,147.057238
7,8,15,1736.818378,1038.722289,0.968400,114.262632,2175.786351,1309.452816,0.947530,94.095962


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# the previous 89 config
numeric_features = ["year", "mileage", "engineSize", "tax", "mpg"]

# -----------------------------
# Helpers (métricas em €)
# -----------------------------
def _rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def _bias(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

# -----------------------------
# CONFIG (a tua)
# -----------------------------
RANDOM_STATE = 42

final_config = {
    "n_estimators": 2000,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.5,
    "max_depth": 20,
    "bootstrap": True,
}

N_SPLITS = 8
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]
low_card_curr = low_card_features  # sem engine_bin

DROP_FROM_MODEL = ["previousOwners"]
numeric_features_model = ["mileage", "engineSize", "tax", "mpg", "age"]  # year -> age

print(f"\n--- 8-FOLD CV | RF (log target) | AGE only | no previousOwners | params={final_config} ---\n")

rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    # -----------------------------
    # Split
    # -----------------------------
    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val   = y.iloc[val_idx].copy()

    # log-target só no treino
    y_train_log = np.log1p(y_train.astype(float))

    # -----------------------------
    # String normalization (fold-wise)
    # -----------------------------
    for col in cols_to_normalize:
        if col in X_train.columns:
            X_train[col] = fill_unknown(X_train[col])
            X_train = column_string_transformer(
                X_train, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
        if col in X_val.columns:
            X_val[col] = fill_unknown(X_val[col])
            X_val = column_string_transformer(
                X_val, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )

    # Restrict to expected schema
    expected_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
    X_train = X_train[expected_cols].copy()
    X_val   = X_val.reindex(columns=expected_cols, fill_value=np.nan).copy()

    # -----------------------------
    # Fit + transform (NO LEAKAGE)
    # -----------------------------
    year_state = fit_year_median(X_train, year_col="year", model_col="model")
    X_train = transform_year_with_model_median(X_train, state=year_state)
    X_val   = transform_year_with_model_median(X_val,   state=year_state)

    mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
    X_train = transform_mileage_imputer(X_train, state=mileage_state)
    X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

    engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
    X_train = transform_engine_size_imputer(X_train, state=engine_state)
    X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

    X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
    X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

    mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
    X_train = transform_mpg_imputer(X_train, state=mpg_state)
    X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

    brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
    X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
    X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

    model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
    X_train, _, _ = transform_invalid_models(X_train, model_state)
    X_val,   _, _ = transform_invalid_models(X_val,   model_state)

    transm_state = fit_transmission_resolver(X_train, valid_transmissions)
    X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
    X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

    fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
    X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
    X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

    # -----------------------------
    # Feature engineering: ONLY AGE (drop year)
    # -----------------------------
    X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
    X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

    # hard guarantee: previousOwners is not used
    X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
    X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

    # -----------------------------
    # Encoding (fit no train com LOG target)
    # -----------------------------
    te = MyTargetEncoder(smoothing=5)
    te.fit(X_train[high_card_features], y_train_log)
    X_train_high = te.transform(X_train[high_card_features])
    X_val_high   = te.transform(X_val[high_card_features])

    if len(low_card_curr) > 0:
        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])
    else:
        X_train_low = pd.DataFrame(index=X_train.index)
        X_val_low   = pd.DataFrame(index=X_val.index)

    X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
    X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

    drop_for_numeric = set(high_card_features + low_card_curr)
    numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]
    numeric_features_curr = [c for c in numeric_features_model if c in numeric_features_curr]

    X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
    X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

    X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

    # ---- imprime features usadas neste fold
    feat_names = list(X_train_final.columns)
    print(f"\n[Fold {fold}] Features used ({len(feat_names)}):")
    print(feat_names)

    # -----------------------------
    # Train RF (log target) + predict (back to €)
    # -----------------------------
    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **final_config
    )
    rf.fit(X_train_final, y_train_log)

    pred_tr_log  = rf.predict(X_train_final)
    pred_val_log = rf.predict(X_val_final)

    pred_tr  = np.expm1(pred_tr_log)
    pred_val = np.expm1(pred_val_log)

    # -----------------------------
    # Metrics (em €)
    # -----------------------------
    tr_rmse = _rmse(y_train, pred_tr)
    tr_mae  = float(mean_absolute_error(y_train, pred_tr))
    tr_r2   = float(r2_score(y_train, pred_tr))
    tr_bias = _bias(y_train, pred_tr)

    val_rmse = _rmse(y_val, pred_val)
    val_mae  = float(mean_absolute_error(y_val, pred_val))
    val_r2   = float(r2_score(y_val, pred_val))
    val_bias = _bias(y_val, pred_val)

    print(
        f"[Fold {fold}] "
        f"TRAIN -> RMSE: {tr_rmse:.1f} | MAE: {tr_mae:.1f} | R2: {tr_r2:.4f} | Bias: {tr_bias:.1f}   ||   "
        f"VAL -> RMSE: {val_rmse:.1f} | MAE: {val_mae:.1f} | R2: {val_r2:.4f} | Bias: {val_bias:.1f}"
    )

    rows.append({
        "fold": fold,
        "n_features": int(X_train_final.shape[1]),
        "train_rmse": tr_rmse, "train_mae": tr_mae, "train_r2": tr_r2, "train_bias": tr_bias,
        "val_rmse": val_rmse, "val_mae": val_mae, "val_r2": val_r2, "val_bias": val_bias,
    })

# -----------------------------
# Summary
# -----------------------------
results_df = pd.DataFrame(rows)

print("\n--- CV SUMMARY (mean ± std) ---")
for split in ["train", "val"]:
    print(
        f"{split.upper()} | "
        f"RMSE: {results_df[f'{split}_rmse'].mean():.1f} ± {results_df[f'{split}_rmse'].std(ddof=1):.1f} | "
        f"MAE: {results_df[f'{split}_mae'].mean():.1f} ± {results_df[f'{split}_mae'].std(ddof=1):.1f} | "
        f"R2: {results_df[f'{split}_r2'].mean():.4f} ± {results_df[f'{split}_r2'].std(ddof=1):.4f} | "
        f"Bias: {results_df[f'{split}_bias'].mean():.1f} ± {results_df[f'{split}_bias'].std(ddof=1):.1f}"
    )

display(results_df)



--- 8-FOLD CV | RF (log target) | AGE only | no previousOwners | params={'n_estimators': 2000, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True} ---


[Fold 1] Features used (15):
['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
[Fold 1] TRAIN -> RMSE: 1136.2 | MAE: 660.9 | R2: 0.9865 | Bias: 82.8   ||   VAL -> RMSE: 1996.1 | MAE: 1248.2 | R2: 0.9560 | Bias: 81.7

[Fold 2] Features used (15):
['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
[Fold 2] TRAIN -> RMSE: 1125.1 | MAE: 659.0 | R2: 0.9867 | Bias: 82.3   ||   VAL -> RMSE: 2169.7 | MAE: 1284.6 | R2: 0.9498 | Bi

,fold,n_features,train_rmse,train_mae,train_r2,train_bias,val_rmse,val_mae,val_r2,val_bias
0,1,15,1136.207201,660.866788,0.986471,82.847113,1996.145328,1248.205263,0.955972,81.725714
1,2,15,1125.072170,658.991863,0.986668,82.271389,2169.731538,1284.570138,0.949824,111.820498
2,3,15,1146.534523,657.910649,0.986153,82.573237,2255.284422,1272.827941,0.945847,163.781832
3,4,15,1069.635249,656.114030,0.987786,81.029714,2742.304974,1302.721140,0.926793,169.299161
4,5,15,1130.992831,659.082592,0.986479,82.043230,2172.851959,1260.272128,0.950923,134.464686
5,6,15,1140.571754,659.189745,0.986246,83.185131,2163.458711,1259.486646,0.951440,157.590876
6,7,15,1134.839018,661.338874,0.986420,82.494796,2105.200886,1239.369291,0.953155,142.920636
7,8,15,1134.578918,661.245520,0.986515,83.293361,2037.136176,1229.788861,0.954004,95.466272


## Previous attempts 

- We conducted a more exaustive search in the beggining of our project. However, we found some errors (such as having included paint quality or using a custom random forest model selector for fs instead of the scikit learn's implementation, for instance). We obtained some good configuration results with lower RMSE than the ones tested here. We decided to try it out and see if it would still make the model perform better. This section contains "proof" of that random search, where the configuration was obtained.